# LeetCode 224 — Basic Calculator (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def calculate(self, s: str) -> int:

```

Bài này yêu cầu ta tự viết một **"máy tính bỏ túi"**: nhận một chuỗi như `"(1+(4+5+2)-3)+(6+8)"` và trả về con số `23` — **không được dùng `eval()`**. Nghe thì đơn giản (ai cũng tính được bằng tay!), nhưng để **máy** đọc từng ký tự và hiểu đâu là số, đâu là dấu, dấu trừ nào áp lên cả nhóm ngoặc... là một kỹ năng thật sự: **parsing** (phân tích cú pháp — đọc một chuỗi ký tự và dựng lại cấu trúc của nó). Đây chính là việc mà trình biên dịch, Excel, máy tính Casio... đều làm.

Bài cũng là "sân tập" kinh điển cho **stack** (ngăn xếp) và **đệ quy**: cấu trúc *ngoặc lồng nhau* gần như luôn được giải bằng một trong hai công cụ này.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đề thật kỹ: dấu trừ **một ngôi** (unary minus, như `-5`, `-(2+3)`) khác dấu trừ **hai ngôi** (binary minus, như `7 - 5`) thế nào, và nó chỉ được xuất hiện ở đâu.
2. **Cách 0 — `eval()`**: vì sao bị cấm, nhưng vẫn rất có ích làm **trọng tài** (oracle) khi kiểm thử.
3. Nền tảng: đọc chuỗi từng ký tự, ghép số nhiều chữ số bằng `num = num * 10 + d`, **tokenize** (tách chuỗi thành số / toán tử / ngoặc).
4. Bước đệm: tính biểu thức **không có ngoặc** chỉ bằng hai biến `result` và `sign` — vì "trừ một số = cộng số đối của nó".
5. **Cách 1 — Brute force "khử ngoặc trong cùng"** giống hệt cách ta tính tay: đúng, nhưng `O(n²)`.
6. **Cách 2 — Stack lưu `(result, sign)` khi gặp `'('`** (lời giải chính, `O(n)`), trace đầy đủ trên `"(1+(4+5+2)-3)+(6+8)"`, `"-(2+3)"` và `"1-(     -2)"`.
7. **Cách 3 — Stack chỉ lưu dấu**: "phân phối" dấu trừ vào trong ngoặc, ví dụ `1-(2-(3+4)) = 1-2+3+4`.
8. **Cách 4 — Đệ quy (recursive descent)**: mỗi cặp ngoặc là một lời gọi hàm; và giới hạn độ sâu đệ quy của Python.
9. **Cách 5 — Shunting-yard → ký pháp hậu tố (RPN)** rồi tính bằng stack; xử lý dấu trừ một ngôi bằng mẹo "chèn số 0".
10. `class Solution` để nộp, bộ test, **đối chiếu chéo cả 5 cách với `eval`** trên 3000 biểu thức ngẫu nhiên, lỗi hay gặp, mẹo nhận dạng, bài liên quan và 6 bài tập có đáp án.

Lộ trình: `eval (cấm)` → `tokenize + result/sign` → `khử ngoặc O(n²)` → `Stack (result, sign) O(n)` → `Stack dấu` → `Đệ quy` → `Shunting-yard + RPN`.

## 1. Đề bài (theo LeetCode)

Cho một chuỗi `s` biểu diễn một **biểu thức hợp lệ** (valid expression), hãy cài đặt một máy tính cơ bản để tính giá trị của biểu thức đó và trả về kết quả.

**Lưu ý:** Bạn **không được phép** dùng bất kỳ hàm có sẵn nào tính giá trị của chuỗi như một biểu thức toán học, chẳng hạn `eval()`.

**Example 1**

```text
Input: s = "1 + 1"
Output: 2
```

**Example 2**

```text
Input: s = " 2-1 + 2 "
Output: 3
```

**Example 3**

```text
Input: s = "(1+(4+5+2)-3)+(6+8)"
Output: 23
```

**Constraints**

- `1 <= s.length <= 3 * 10^5`
- `s` consists of digits, `'+'`, `'-'`, `'('`, `')'`, and `' '`.
- `s` represents a valid expression.
- `'+'` is not used as a unary operation (i.e., `"+1"` and `"+(2 + 3)"` is invalid).
- `'-'` could be used as a unary operation (i.e., `"-1"` and `"-(2 + 3)"` is valid).
- There will be no two consecutive operators in the input.
- Every number and running calculation will fit in a signed 32-bit integer.

**Dịch nhanh các ràng buộc:**

- Độ dài chuỗi từ 1 tới 300 000 ký tự.
- Chỉ có 5 loại ký tự: chữ số, `+`, `-`, `(`, `)` và khoảng trắng. **Không có** `*`, `/`.
- Chuỗi luôn là biểu thức hợp lệ (ngoặc cân bằng, không thiếu số...).
- `'+'` **không bao giờ** đứng một mình trước số kiểu `"+1"`.
- `'-'` **có thể** là dấu một ngôi: `"-1"`, `"-(2 + 3)"`.
- Không có hai toán tử đứng liền nhau (kiểu `"1 + -2"` là không có).
- Mọi số và mọi kết quả trung gian đều nằm trong số nguyên có dấu 32-bit (từ `-2^31` tới `2^31 - 1`).

## 2. Đọc kỹ đề: cái gì được phép, cái gì không?

### Toán tử hai ngôi và một ngôi

- **Toán tử hai ngôi** (binary operator) cần **hai** toán hạng (operand) ở hai bên: `7 - 5` → lấy 7 trừ 5.
- **Toán tử một ngôi** (unary operator) chỉ cần **một** toán hạng đứng sau: `-5` → "số đối của 5"; `-(2 + 3)` → "số đối của cả nhóm".

Cùng là ký tự `'-'`, nhưng nghĩa khác nhau tùy vị trí. Bảng dưới giúp ta hình dung chuỗi nào là đầu vào hợp lệ:

| Chuỗi | Hợp lệ theo đề? | Lý do |
|---|:-:|---|
| `"1 + 1"` | Có | phép cộng hai ngôi bình thường |
| `"-1"` | Có | `'-'` một ngôi ở đầu biểu thức |
| `"-(2 + 3)"` | Có | `'-'` một ngôi đứng trước cả nhóm ngoặc |
| `"1-(     -2)"` | Có | `'-'` một ngôi ngay sau `'('` (khoảng trắng không tính) |
| `"+1"` | Không | `'+'` không được dùng một ngôi |
| `"+(2 + 3)"` | Không | như trên |
| `"1 + -2"` | Không | hai toán tử liên tiếp `+` rồi `-` |
| `"2 * 3"` | Không | không có ký tự `*` trong bài này |
| `"()"` | Không | trong ngoặc phải là một biểu thức, không được rỗng |

### Hệ quả quan trọng (sẽ dùng lại ở Cách 5)

Vì `'+'` không bao giờ là một ngôi và không có hai toán tử liền nhau, **dấu `'-'` một ngôi chỉ có thể đứng ở 2 chỗ**:

1. Ở **đầu** biểu thức (`"-1 + 2"`), hoặc
2. **Ngay sau `'('`** (`"1-(-2)"`), bỏ qua khoảng trắng.

Cách nhận biết: nhìn ký tự **khác khoảng trắng** đứng ngay trước dấu `'-'`. Không có gì (đầu chuỗi) hoặc là `'('` → một ngôi; còn lại (một chữ số hoặc `')'`) → hai ngôi.

### Kích thước: cần thuật toán gần tuyến tính

`n` có thể tới `3 * 10^5`. Python thuần chạy cỡ `10^7` phép tính đơn giản mỗi giây. Một thuật toán `O(n²)` sẽ tốn cỡ `(3 * 10^5)^2 = 9 * 10^10` bước → hàng giờ đồng hồ. Mục tiêu: **`O(n)`** — đọc chuỗi một (vài) lượt.

### Về giới hạn 32-bit

Số nguyên Python không bao giờ tràn, nên ta không phải lo. Nhưng nếu bạn viết bằng C++/Java thì đây là lời đảm bảo rằng dùng `int` là đủ.

Cell dưới in các ví dụ bằng `repr` (dạng `{s!r}` trong f-string: in chuỗi **kèm dấu nháy** để thấy rõ khoảng trắng đầu/cuối), kiểm tra chúng chỉ chứa ký tự cho phép bằng **tập hợp** (`set`, phép `<=` nghĩa là "là tập con của"), rồi viết hàm nhỏ `minus_kinds` phân loại từng dấu `'-'`. Hàm dùng `enumerate(s)` — cho ra lần lượt từng cặp `(chỉ số, ký tự)`.

In [1]:
examples = [
    ("1 + 1", 2),
    (" 2-1 + 2 ", 3),
    ("(1+(4+5+2)-3)+(6+8)", 23),
]
ALLOWED = set("0123456789+-() ")

for s, expected in examples:
    print(f"s = {s!r:24} kỳ vọng = {expected:3}   chỉ gồm ký tự hợp lệ? {set(s) <= ALLOWED}")


def minus_kinds(s: str) -> list:
    """Với mỗi dấu '-', nhìn ký tự KHÁC khoảng trắng đứng ngay trước nó."""
    kinds = []
    prev = None                      # chưa thấy ký tự nào
    for i, ch in enumerate(s):
        if ch == " ":
            continue                 # khoảng trắng: bỏ qua, không cập nhật prev
        if ch == "-":
            kind = "một ngôi" if prev is None or prev == "(" else "hai ngôi"
            kinds.append((i, kind))
        prev = ch
    return kinds

print()
print(minus_kinds("- (3 + (4 + 5))"))
print(minus_kinds("1-(     -2)"))
print(minus_kinds("10 - 2 - (-3)"))

s = '1 + 1'                  kỳ vọng =   2   chỉ gồm ký tự hợp lệ? True
s = ' 2-1 + 2 '              kỳ vọng =   3   chỉ gồm ký tự hợp lệ? True
s = '(1+(4+5+2)-3)+(6+8)'    kỳ vọng =  23   chỉ gồm ký tự hợp lệ? True

[(0, 'một ngôi')]
[(1, 'hai ngôi'), (8, 'một ngôi')]
[(3, 'hai ngôi'), (7, 'hai ngôi'), (10, 'một ngôi')]


## 3. Cách 0 — `eval()`: vì sao bị cấm, và vì sao ta vẫn dùng nó (để test)

`eval(chuỗi)` là hàm có sẵn của Python: nó coi `chuỗi` là **một đoạn code Python**, chạy đoạn code đó và trả về kết quả. Tình cờ là cú pháp số nguyên, `+`, `-`, ngoặc của Python trùng với cú pháp của đề, nên `eval(s)` cho đáp án đúng với các ví dụ. Vậy tại sao đề cấm?

1. **Mục tiêu của bài là tự viết bộ phân tích biểu thức.** Dùng `eval` là "nhờ Python làm hộ" phần khó nhất — vi phạm yêu cầu đề; trong phỏng vấn coi như không giải.
2. **Nguy hiểm về bảo mật.** `eval` chạy **bất kỳ** code Python nào chứ không chỉ phép tính. Nếu chuỗi đến từ người dùng (ví dụ ô nhập công thức trên web), họ có thể gõ một lệnh xóa file hay đọc mật khẩu. "Không bao giờ `eval` dữ liệu không tin cậy" là quy tắc vàng của lập trình.
3. **Ngữ pháp Python ≠ ngữ pháp đề.** Python báo lỗi với số có chữ số 0 ở đầu như `"007"`; ngược lại Python chấp nhận `"2*3"`, `"2**10"`, `"+1"`... mà đề không hề có. "Gần giống" chưa phải là "đúng".

Cell dưới cho thấy cả ba điều. Nó dùng `try/except`: Python **thử** chạy khối `try`; nếu xảy ra lỗi loại `SyntaxError` thì nhảy vào khối `except` (in thông báo) thay vì dừng cả notebook.

In [2]:
print(eval("1 + 1"))
print(eval(" 2-1 + 2 "))
print(eval("(1+(4+5+2)-3)+(6+8)"))
print(eval("-(2 + 3)"), eval("1-(     -2)"))

# eval chạy BẤT KỲ code Python nào, không chỉ phép tính:
print(eval("len('abc') + 100"))      # 103 — đây là GỌI HÀM, không phải toán!

# Ngữ pháp Python khác ngữ pháp đề: số có chữ số 0 ở đầu
try:
    eval("007 + 1")
except SyntaxError as e:
    print("SyntaxError:", e.msg)

2
3
23
-5 3
103
SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers


### `eval` làm **trọng tài** (oracle) khi kiểm thử

Trong kiểm thử phần mềm, **oracle** ("nhà tiên tri") là một cách tính mà ta **tin là đúng** (thường chậm hoặc "ăn gian") dùng để so kết quả với lời giải thật. Ở đây `eval` là oracle tuyệt vời: ta sẽ sinh hàng nghìn biểu thức ngẫu nhiên hợp lệ và so mọi lời giải với `eval`. Quy ước của notebook:

- `oracle(s)` **chỉ dùng trong test**, không bao giờ nộp.
- Trước khi `eval`, kiểm tra chuỗi **chỉ chứa ký tự cho phép** (whitelist) — thói quen an toàn tối thiểu khi buộc phải dùng `eval`.

`assert điều_kiện, thông_báo` nghĩa là: nếu `điều_kiện` sai thì dừng ngay với lỗi `AssertionError` kèm thông báo; nếu đúng thì không làm gì. Đây là cách viết test gọn nhất.

In [3]:
def oracle(s: str) -> int:
    """Trọng tài CHỈ dùng để test: kiểm tra whitelist rồi mới eval."""
    assert set(s) <= ALLOWED, f"ký tự lạ trong {s!r}"
    return eval(s)

for s, expected in examples:
    assert oracle(s) == expected
print("oracle khớp cả 3 ví dụ của đề")

oracle khớp cả 3 ví dụ của đề


## 4. Kiến thức nền 1 — Đọc chuỗi từng ký tự và ghép số nhiều chữ số

Máy tính **không "nhìn" thấy số 345** trong chuỗi `"345+1"` — nó chỉ thấy lần lượt các ký tự `'3'`, `'4'`, `'5'`, `'+'`, `'1'`. Việc đầu tiên là **phân loại** từng ký tự:

| Kiểm tra | Ý nghĩa |
|---|---|
| `ch.isdigit()` | `True` nếu `ch` là chữ số `'0'`..`'9'` |
| `ch == " "` | khoảng trắng |
| `ch in "+-"` | toán tử (`in` trên chuỗi: `ch` có nằm trong `"+-"` không) |
| `ch == "("`, `ch == ")"` | ngoặc mở / ngoặc đóng |

### Ghép số: `num = num * 10 + d`

Đọc `"345"` từ trái sang phải, mỗi khi gặp chữ số mới `d`:

```text
ban đầu        num = 0
đọc '3' (d=3): num = 0  * 10 + 3 = 3
đọc '4' (d=4): num = 3  * 10 + 4 = 34
đọc '5' (d=5): num = 34 * 10 + 5 = 345
```

**Vì sao nhân 10?** Nhân 10 đẩy mọi chữ số cũ lên một hàng (hàng đơn vị → hàng chục, hàng chục → hàng trăm), chừa chỗ trống ở hàng đơn vị cho chữ số mới. (Đây đúng là công thức `reversed_num * 10 + digit` trong bài Palindrome Number, chỉ khác là ta đọc xuôi.)

### Đổi ký tự `'7'` thành số `7`

- `int("7")` → `7`.
- Hoặc dùng mã ASCII: `ord("7") - ord("0")` → `55 - 48 = 7` (các chữ số có mã liên tiếp nhau).

**Cẩn thận:** `"3" + "4"` là **nối chuỗi** ra `"34"`, còn `3 + 4` mới là `7`. Phải đổi sang `int` trước khi tính.

In [4]:
for ch in "12 + (3-4)":
    if ch.isdigit():
        kind = "chữ số"
    elif ch == " ":
        kind = "khoảng trắng"
    elif ch in "+-":
        kind = "toán tử"
    else:
        kind = "ngoặc"
    print(repr(ch), "->", kind)

print()
num = 0
for ch in "345":
    d = int(ch)                   # hoặc ord(ch) - ord("0")
    num = num * 10 + d
    print(f"đọc {ch!r} -> num = {num}")
print("'3' + '4' =", "3" + "4", "   nhưng 3 + 4 =", 3 + 4)
print("ord('7') - ord('0') =", ord("7") - ord("0"))

'1' -> chữ số
'2' -> chữ số
' ' -> khoảng trắng
'+' -> toán tử
' ' -> khoảng trắng
'(' -> ngoặc
'3' -> chữ số
'-' -> toán tử
'4' -> chữ số
')' -> ngoặc

đọc '3' -> num = 3
đọc '4' -> num = 34
đọc '5' -> num = 345
'3' + '4' = 34    nhưng 3 + 4 = 7
ord('7') - ord('0') = 7


## 5. Kiến thức nền 2 — Tokenize: tách chuỗi thành các "từ" (token)

**Token** là đơn vị nhỏ nhất có nghĩa — giống "từ" trong một câu. Bước tách chuỗi thành token gọi là **tokenize** (hay *lexing*). Ví dụ:

```text
"12 + (3-45)"   ->   [12, '+', '(', 3, '-', 45, ')']
```

- Khoảng trắng bị **bỏ đi** (nó không mang nghĩa toán học).
- Số nhiều chữ số `"12"`, `"45"` thành **một** token kiểu `int`.
- Toán tử và ngoặc giữ nguyên là chuỗi 1 ký tự.

### Thuật toán (giả mã)

```text
i = 0
trong khi i < n:
    nếu s[i] là ' '      : i += 1                        (bỏ qua)
    nếu s[i] là chữ số   : vòng lặp con đọc HẾT các chữ số liền nhau,
                           ghép thành num, thêm num vào tokens
    còn lại (+ - ( ))    : thêm s[i] vào tokens, i += 1
```

**Vì sao dùng `while` với chỉ số `i` tự quản lý, không dùng `for ch in s`?** Vì khi gặp một số, ta phải **nhảy qua nhiều ký tự một lúc** (đọc `"12"` xong thì `i` nhảy từ 0 sang 2). Vòng `for` luôn đi đúng một bước, không cho ta nhảy.

**Điều kiện vòng lặp con:** `while i < n and s[i].isdigit()`. Phải kiểm tra `i < n` **trước**: Python đánh giá `and` theo kiểu *short-circuit* (ngắn mạch) — nếu vế trái đã `False` thì không xét vế phải, nhờ vậy không bao giờ đọc `s[n]` (vượt biên) khi số nằm ở cuối chuỗi.

Cell dưới viết `tokenize`, cell sau nữa là bản `trace_tokenize` in từng lượt của vòng `while` lớn: hãy để ý cột `i` nhảy cóc khi gặp số nhiều chữ số.

In [5]:
def tokenize(s: str) -> list:
    tokens = []
    i, n = 0, len(s)
    while i < n:
        ch = s[i]
        if ch == " ":
            i += 1                                   # bỏ qua khoảng trắng
        elif ch.isdigit():
            num = 0
            while i < n and s[i].isdigit():          # đọc hết các chữ số liền nhau
                num = num * 10 + int(s[i])
                i += 1
            tokens.append(num)                       # i đã đứng ở ký tự SAU số
        else:                                        # '+', '-', '(', ')'
            tokens.append(ch)
            i += 1
    return tokens

print(tokenize("12 + (3-45)"))
for s, _ in examples:
    print(f"{s!r:24} -> {tokenize(s)}")

[12, '+', '(', 3, '-', 45, ')']
'1 + 1'                  -> [1, '+', 1]
' 2-1 + 2 '              -> [2, '-', 1, '+', 2]
'(1+(4+5+2)-3)+(6+8)'    -> ['(', 1, '+', '(', 4, '+', 5, '+', 2, ')', '-', 3, ')', '+', '(', 6, '+', 8, ')']


In [6]:
def trace_tokenize(s: str) -> list:
    tokens = []
    i, n = 0, len(s)
    while i < n:
        ch = s[i]
        if ch == " ":
            print(f"i={i:2} ch={ch!r}  -> khoảng trắng, bỏ qua")
            i += 1
        elif ch.isdigit():
            start, num = i, 0
            while i < n and s[i].isdigit():
                num = num * 10 + int(s[i])
                i += 1
            tokens.append(num)
            print(f"i={start:2} ch={ch!r}  -> đọc liền {s[start:i]!r}, i nhảy tới {i}   tokens = {tokens}")
        else:
            tokens.append(ch)
            print(f"i={i:2} ch={ch!r}  -> toán tử/ngoặc              tokens = {tokens}")
            i += 1
    return tokens

assert trace_tokenize("12 + (3-45)") == tokenize("12 + (3-45)")

i= 0 ch='1'  -> đọc liền '12', i nhảy tới 2   tokens = [12]
i= 2 ch=' '  -> khoảng trắng, bỏ qua
i= 3 ch='+'  -> toán tử/ngoặc              tokens = [12, '+']
i= 4 ch=' '  -> khoảng trắng, bỏ qua
i= 5 ch='('  -> toán tử/ngoặc              tokens = [12, '+', '(']
i= 6 ch='3'  -> đọc liền '3', i nhảy tới 7   tokens = [12, '+', '(', 3]
i= 7 ch='-'  -> toán tử/ngoặc              tokens = [12, '+', '(', 3, '-']
i= 8 ch='4'  -> đọc liền '45', i nhảy tới 10   tokens = [12, '+', '(', 3, '-', 45]
i=10 ch=')'  -> toán tử/ngoặc              tokens = [12, '+', '(', 3, '-', 45, ')']


## 6. Bước đệm — Tính biểu thức KHÔNG có ngoặc bằng `result` và `sign`

Trước khi đụng tới ngoặc, hãy giải bài nhỏ hơn: `"2 - 1 + 2"` tính thế nào khi đọc từ trái sang phải?

**Mấu chốt toán học: trừ một số = cộng số đối của nó.**

```text
2 - 1 + 2  =  (+2) + (-1) + (+2)  =  3
```

Vậy biểu thức chỉ có `+`, `-` thực chất là **một tổng các hạng tử có dấu** (signed terms). Ta chỉ cần 2 biến:

- `result`: tổng các hạng tử **đã đọc xong**.
- `sign`: dấu (`+1` hoặc `-1`) đứng **trước hạng tử sắp đọc**. Ban đầu `sign = +1` (số đầu tiên không có dấu thì là số dương).

Quy tắc: gặp `'+'` → `sign = +1`; gặp `'-'` → `sign = -1`; gặp số `x` → `result += sign * x`.

### Tính tay với `"-5 + 12 - 3"` (tokens: `['-', 5, '+', 12, '-', 3]`)

| Token | `sign` sau bước | `result` sau bước | Giải thích |
|:-:|:-:|:-:|---|
| `'-'` | -1 | 0 | dấu trừ **một ngôi**: chỉ đổi dấu cho số sắp tới |
| `5` | -1 | -5 | `result += (-1) * 5` |
| `'+'` | +1 | -5 | |
| `12` | +1 | 7 | `result += (+1) * 12` |
| `'-'` | -1 | 7 | |
| `3` | -1 | 4 | `result += (-1) * 3` |

Kết quả `4` (đúng: `-5 + 12 - 3 = 4`).

**Điều thú vị:** dấu trừ một ngôi ở đầu được xử lý **"miễn phí"**, không cần `if` riêng! Lý do: `-5` chính là `0 - 5`, và `result` vốn bắt đầu bằng `0`.

Vì list `tokens` trộn lẫn `int` và chuỗi, ta dùng `isinstance(tok, int)` để hỏi "`tok` có phải số nguyên không?".

In [7]:
def eval_flat_tokens(tokens: list) -> int:
    """Tính list token KHÔNG có ngoặc, ví dụ ['-', 5, '+', 12, '-', 3]."""
    result, sign = 0, 1
    for tok in tokens:
        if isinstance(tok, int):
            result += sign * tok
        elif tok == "+":
            sign = 1
        else:                       # tok == '-'
            sign = -1
    return result


def trace_flat(s: str) -> int:
    result, sign = 0, 1
    for tok in tokenize(s):
        if isinstance(tok, int):
            result += sign * tok
            print(f"token {tok!r:4} -> result += ({sign:+d})*{tok}  => result = {result}")
        else:
            sign = 1 if tok == "+" else -1
            print(f"token {tok!r:4} -> sign = {sign:+d}")
    return result

print("Kết quả:", trace_flat("-5 + 12 - 3"))
print(eval_flat_tokens(tokenize(" 2-1 + 2 ")), eval_flat_tokens(tokenize("1 + 1")))

token '-'  -> sign = -1
token 5    -> result += (-1)*5  => result = -5
token '+'  -> sign = +1
token 12   -> result += (+1)*12  => result = 7
token '-'  -> sign = -1
token 3    -> result += (-1)*3  => result = 4
Kết quả: 4
3 2


### 6.1 Gộp tokenize và tính toán vào **một lượt** duyệt — mẫu "chốt số" (flush)

Thật ra ta không cần tạo list `tokens` riêng. Duyệt từng ký tự bằng `for ch in s`, và:

- Gặp **chữ số**: ghép vào `num` (`num = num * 10 + d`). Số chưa xong nên **chưa cộng** vào `result`.
- Gặp **toán tử** `'+'`/`'-'`: số đứng trước nó chắc chắn đã đọc xong → **chốt** (flush) nó: `result += sign * num`, đặt lại `num = 0`, rồi mới cập nhật `sign` theo toán tử vừa gặp.
- Gặp **khoảng trắng**: không làm gì cả.
- **Hết chuỗi**: số cuối cùng **không có toán tử nào phía sau** để kích hoạt việc chốt → phải chốt lần cuối: `return result + sign * num`.

> Quên lần chốt cuối là lỗi **số 1** của bài này: `"1 + 1"` sẽ ra `1`. (Xem mục Lỗi thường gặp.)

Còn dấu `'-'` một ngôi ở đầu? Khi gặp nó, ta "chốt" `num = 0` → `result += (+1) * 0` — cộng thêm 0, hoàn toàn vô hại — rồi đặt `sign = -1`. Một lần nữa, **không cần trường hợp đặc biệt**.

Hàm `eval_no_paren` dưới đây chính là **bộ xương** của lời giải chính: Cách 2 chỉ thêm hai nhánh cho `'('` và `')'`.

In [8]:
def eval_no_paren(s: str) -> int:
    """Tính biểu thức chỉ có số, '+', '-', khoảng trắng (chưa hiểu ngoặc)."""
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)          # đang ghép số, CHƯA cộng
        elif ch in "+-":
            result += sign * num              # chốt số vừa đọc xong
            num = 0
            sign = 1 if ch == "+" else -1     # dấu cho số TIẾP THEO
        # khoảng trắng: không làm gì
    return result + sign * num                # chốt số cuối cùng!

for s in ["1 + 1", " 2-1 + 2 ", "-5 + 12 - 3", "42", "100 - 99 - 1"]:
    print(f"{s!r:16} -> {eval_no_paren(s):4}   (oracle: {oracle(s)})")
    assert eval_no_paren(s) == oracle(s)

'1 + 1'          ->    2   (oracle: 2)
' 2-1 + 2 '      ->    3   (oracle: 3)
'-5 + 12 - 3'    ->    4   (oracle: 4)
'42'             ->   42   (oracle: 42)
'100 - 99 - 1'   ->    0   (oracle: 0)


## 7. Vấn đề: dấu ngoặc làm hỏng mọi thứ

Thử `"1 - (2 + 3)"`. Đúng phải là `1 - 5 = -4`.

Nếu "lờ" ngoặc đi và đọc trái sang phải như mục 6, ta tính `1 - 2 + 3 = 2` — **sai**! Lý do: dấu `'-'` trước ngoặc áp lên **cả nhóm** `(2 + 3)`, chứ không chỉ lên số `2`:

```text
1 - (2 + 3)  =  1 - 2 - 3      (dấu '+' bên trong bị "lật" thành '-')
1 - (2 - 3)  =  1 - 2 + 3      (dấu '-' bên trong bị "lật" thành '+')
```

Nói cách khác: **trong lúc tính phần bên trong ngoặc, ta phải nhớ được phần bên ngoài** (kết quả đã có, và dấu đứng trước ngoặc). Toàn bộ các cách giải phía sau chỉ là những cách khác nhau để "nhớ" điều đó.

Cell dưới cho `eval_no_paren` chạy trên chuỗi có ngoặc (nó bỏ qua `'('`, `')'` vì không có nhánh nào khớp) và so với oracle.

In [9]:
for s in ["1 - (2 + 3)", "1 - (2 - 3)", "(1+(4+5+2)-3)+(6+8)", "-(2 + 3)"]:
    wrong = eval_no_paren(s)          # lờ ngoặc đi
    print(f"{s!r:24} lờ ngoặc -> {wrong:3}   đúng phải là {oracle(s):3}   {'OK' if wrong == oracle(s) else 'SAI'}")

'1 - (2 + 3)'            lờ ngoặc ->   2   đúng phải là  -4   SAI
'1 - (2 - 3)'            lờ ngoặc ->  -4   đúng phải là   2   SAI
'(1+(4+5+2)-3)+(6+8)'    lờ ngoặc ->  23   đúng phải là  23   OK
'-(2 + 3)'               lờ ngoặc ->   1   đúng phải là  -5   SAI


## 8. Cách 1 — Brute force: khử cặp ngoặc trong cùng, lặp lại (như tính tay)

Hồi phổ thông, ta tính `(1+(4+5+2)-3)+(6+8)` thế này:

```text
Bước 1: ngoặc trong cùng (4+5+2) = 11   ->  (1+11-3)+(6+8)
Bước 2: ngoặc trong cùng (1+11-3) = 9   ->  9+(6+8)
Bước 3: ngoặc trong cùng (6+8) = 14     ->  9+14
Bước 4: hết ngoặc, tính thẳng            ->  23
```

Đây là cách mà người mới **tự nghĩ ra** đầu tiên. Ta dạy máy làm y hệt:

1. **Tìm cặp ngoặc trong cùng:** dấu `')'` **đầu tiên** tính từ trái chắc chắn đóng một cặp **không chứa ngoặc nào bên trong** (nếu bên trong còn `')'` thì nó đã không phải là `')'` đầu tiên). Dấu `'('` tương ứng là `'('` **gần nhất bên trái** nó.
2. Đoạn giữa hai ngoặc là biểu thức không ngoặc → tính bằng `eval_flat_tokens` (mục 6).
3. **Thay** cả đoạn `( ... )` bằng một token duy nhất là giá trị vừa tính. Lặp lại tới khi hết `')'`.

**Vì sao làm trên list token chứ không làm trên chuỗi?** Nếu kết quả âm, ví dụ `"1-(2-5)"`: thay `"(2-5)"` bằng `"-3"` sẽ ra chuỗi `"1--3"` — hai dấu liền nhau, các hàm ở mục 6 sẽ hiểu sai. Trên list token, `-3` là **một `int` duy nhất**: `[1, '-', -3]` → `1 + (-1)*(-3) = 4`, đúng.

**Gán vào lát cắt (slice assignment):** `tokens[a:b+1] = [value]` thay **cả đoạn** từ vị trí `a` tới `b` (tính cả `b`) bằng list một phần tử. List tự co lại — xem ví dụ mini trong cell dưới.

Ở cell trace phía sau, mỗi dòng in list token **trước** một vòng khử ngoặc và giá trị vừa tính ra. Ví dụ thứ hai `"- (3 + (4 - 10))"` cố tình có kết quả trung gian âm và dấu trừ một ngôi trước ngoặc.

In [10]:
demo = ["a", "(", "b", "c", ")", "d"]
demo[1:4 + 1] = ["X"]                  # thay đoạn từ index 1 tới 4 bằng ["X"]
print("slice assignment:", demo)


def calculate_reduce(s: str) -> int:
    """Cách 1: lặp lại việc khử cặp ngoặc trong cùng. O(n^2)."""
    tokens = tokenize(s)
    while ")" in tokens:
        close = tokens.index(")")            # ')' đầu tiên từ trái
        open_ = close - 1
        while tokens[open_] != "(":          # '(' gần nhất bên trái nó
            open_ -= 1
        value = eval_flat_tokens(tokens[open_ + 1:close])
        tokens[open_:close + 1] = [value]    # thay cả "( ... )" bằng 1 số
    return eval_flat_tokens(tokens)

for s, expected in examples:
    print(f"{s!r:24} -> {calculate_reduce(s)}   (kỳ vọng {expected})")

slice assignment: ['a', 'X', 'd']
'1 + 1'                  -> 2   (kỳ vọng 2)
' 2-1 + 2 '              -> 3   (kỳ vọng 3)
'(1+(4+5+2)-3)+(6+8)'    -> 23   (kỳ vọng 23)


In [11]:
def show(tokens: list) -> str:
    return " ".join(map(str, tokens))


def trace_reduce(s: str) -> int:
    tokens = tokenize(s)
    round_no = 1
    while ")" in tokens:
        close = tokens.index(")")
        open_ = close - 1
        while tokens[open_] != "(":
            open_ -= 1
        inner = tokens[open_ + 1:close]
        value = eval_flat_tokens(inner)
        print(f"Vòng {round_no}: {show(tokens):40} | trong cùng: ({show(inner)}) = {value}")
        tokens[open_:close + 1] = [value]
        round_no += 1
    final = eval_flat_tokens(tokens)
    print(f"Hết ngoặc: {show(tokens):38} | kết quả = {final}")
    return final

assert trace_reduce("(1+(4+5+2)-3)+(6+8)") == 23
print()
assert trace_reduce("- (3 + (4 - 10))") == 3

Vòng 1: ( 1 + ( 4 + 5 + 2 ) - 3 ) + ( 6 + 8 )    | trong cùng: (4 + 5 + 2) = 11
Vòng 2: ( 1 + 11 - 3 ) + ( 6 + 8 )               | trong cùng: (1 + 11 - 3) = 9
Vòng 3: 9 + ( 6 + 8 )                            | trong cùng: (6 + 8) = 14
Hết ngoặc: 9 + 14                                 | kết quả = 23

Vòng 1: - ( 3 + ( 4 - 10 ) )                     | trong cùng: (4 - 10) = -6
Vòng 2: - ( 3 + -6 )                             | trong cùng: (3 + -6) = -3
Hết ngoặc: - -3                                   | kết quả = 3


### 8.1 Vì sao Cách 1 chậm?

Hãy đếm công việc của **một** vòng `while`:

- `")" in tokens` và `tokens.index(")")` duyệt list → tới `O(n)`.
- Gán lát cắt `tokens[a:b+1] = [value]` phải **dịch chuyển** toàn bộ phần đuôi list sang trái → tới `O(n)`.

Số vòng = **số cặp ngoặc**, có thể tới khoảng `n / 2` (ví dụ chuỗi lồng sâu `"((((...1...))))"` hay chuỗi nhiều nhóm `"(1)+(1)+(1)+..."`). Tổng cộng: **Time `O(n²)`**, **Space `O(n)`** (list token).

Với `n = 3 * 10^5` và chuỗi lồng sâu cỡ `1.5 * 10^5` mức, số thao tác cỡ `(1.5 * 10^5)^2 ≈ 2 * 10^10` → quá chậm.

Cell dưới đo "công việc" một cách tất định (không phụ thuộc tốc độ máy): cộng dồn `len(tokens)` ở mỗi vòng — xấp xỉ số phần tử list bị chạm. Để ý: **độ sâu tăng gấp đôi thì công việc tăng gấp ~4** — dấu hiệu kinh điển của `O(n²)`.

**Nút thắt:** ta cứ phải **quay lại** quét list để tìm ngoặc. Ý tưởng cải tiến: đọc **một lượt** từ trái sang phải, và khi gặp `'('` thì **cất tạm** những gì cần nhớ — đó là việc của **stack**.

In [12]:
def reduce_work(s: str) -> int:
    """Giống calculate_reduce nhưng chỉ đếm số phần tử list bị chạm."""
    tokens = tokenize(s)
    work = 0
    while ")" in tokens:
        work += len(tokens)                  # mỗi vòng quét/dịch cỡ len(tokens) phần tử
        close = tokens.index(")")
        open_ = close - 1
        while tokens[open_] != "(":
            open_ -= 1
        tokens[open_:close + 1] = [eval_flat_tokens(tokens[open_ + 1:close])]
    return work

for k in [100, 200, 400, 800]:
    s = "(" * k + "1" + ")" * k              # lồng sâu k mức
    w = reduce_work(s)
    print(f"độ sâu k = {k:3}   len(s) = {len(s):4}   công việc = {w:7}   công việc / k² = {w / k**2:.2f}")

độ sâu k = 100   len(s) =  201   công việc =   10200   công việc / k² = 1.02
độ sâu k = 200   len(s) =  401   công việc =   40400   công việc / k² = 1.01
độ sâu k = 400   len(s) =  801   công việc =  160800   công việc / k² = 1.00
độ sâu k = 800   len(s) = 1601   công việc =  641600   công việc / k² = 1.00


## 9. Kiến thức nền 3 — Stack (ngăn xếp)

Hãy tưởng tượng một **chồng đĩa**: đĩa mới luôn đặt lên **trên cùng**, và khi lấy cũng lấy từ **trên cùng**. Đĩa đặt vào **sau cùng** được lấy ra **đầu tiên** — gọi là **LIFO** (*Last In, First Out*: vào sau, ra trước).

Trong Python, `list` dùng làm stack rất tiện:

| Thao tác | Code | Ý nghĩa | Chi phí |
|---|---|---|:-:|
| push | `stack.append(x)` | đặt `x` lên đỉnh | `O(1)` |
| pop | `stack.pop()` | lấy phần tử ở đỉnh ra (và xóa khỏi stack) | `O(1)` |
| peek | `stack[-1]` | xem phần tử ở đỉnh, không xóa | `O(1)` |
| rỗng? | `not stack` | `True` nếu stack rỗng | `O(1)` |

**Vì sao stack hợp với ngoặc?** Trong `"( ( ) )"`, ngoặc mở **sau cùng** luôn được đóng **trước tiên** — đúng tinh thần LIFO. Mỗi lần gặp `'('` ta *push* thông tin của mức ngoài; mỗi lần gặp `')'` ta *pop* đúng thông tin của mức vừa bao quanh.

**Tuple** `(10, -1)` gói 2 giá trị làm một phần tử. Lấy ra và tách bằng **unpacking**: `r, sg = stack.pop()` → `r = 10`, `sg = -1`.

In [13]:
stack = []
stack.append((10, -1))         # push một tuple 2 phần tử
stack.append((5, 1))
print("stack:", stack, "  đỉnh (peek):", stack[-1])

r, sg = stack.pop()            # pop lấy phần tử vào SAU CÙNG
print("pop ra:", (r, sg), "  còn lại:", stack)

r, sg = stack.pop()
print("pop ra:", (r, sg), "  còn lại:", stack, "  rỗng?", not stack)

stack: [(10, -1), (5, 1)]   đỉnh (peek): (5, 1)
pop ra: (5, 1)   còn lại: [(10, -1)]
pop ra: (10, -1)   còn lại: []   rỗng? True


## 10. Cách 2 — Stack lưu `(result, sign)` khi gặp `'('` (lời giải chính)

### 10.1 Mấu chốt: "đóng băng" phần bên ngoài

Ta muốn đọc chuỗi **đúng một lượt** từ trái sang phải với bộ xương `eval_no_paren`. Giả sử đang đọc `"10 - (2 + 3)"` và vừa gặp `'('`. Lúc này ta đã biết:

- `result = 10` — phần bên ngoài **đã tính xong**,
- `sign = -1` — dấu đứng trước ngoặc, sẽ áp lên **cả nhóm**.

Phần trong ngoặc `2 + 3` là **một biểu thức mới hoàn toàn** — tính nó y hệt mục 6, bắt đầu từ `result = 0`, `sign = +1`. Nhưng không được quên `10` và `-1`! → **cất** cặp `(10, -1)` vào stack, rồi reset.

Khi gặp `')'`: chốt số cuối của nhóm → `result` lúc này là **giá trị cả nhóm** (= 5). Lấy `(10, -1)` ra và ghép: `result = 10 + (-1) * 5 = 5`.

**Vì sao cần stack mà không phải 2 biến?** Vì ngoặc có thể **lồng nhau**: `1 - (2 - (3 + 4))`. Mỗi mức ngoặc đang mở cần cất một cặp riêng, và cặp cất **sau** phải được dùng **trước** (ngoặc trong đóng trước ngoặc ngoài) → LIFO → stack.

### Tính tay `"10 - (2 + 3)"` (khoảng trắng bỏ qua, không ghi)

| Bước | Ký tự | Việc làm | `num` | `sign` | `result` | `stack` (sau bước) |
|:-:|:-:|---|:-:|:-:|:-:|---|
| 1 | `1` | `ghép chữ số -> num = 1` | 1 | +1 | 0 | `[]` |
| 2 | `0` | `ghép chữ số -> num = 10` | 10 | +1 | 0 | `[]` |
| 3 | `-` | `chốt +10 vào result, sign = -1` | 0 | -1 | 10 | `[]` |
| 4 | `(` | `cất (10, -1) vào stack, reset` | 0 | +1 | 0 | `[(10, -1)]` |
| 5 | `2` | `ghép chữ số -> num = 2` | 2 | +1 | 0 | `[(10, -1)]` |
| 6 | `+` | `chốt +2 vào result, sign = +1` | 0 | +1 | 2 | `[(10, -1)]` |
| 7 | `3` | `ghép chữ số -> num = 3` | 3 | +1 | 2 | `[(10, -1)]` |
| 8 | `)` | `nhóm = 5; lấy (10, -1) ra: 10 + (-1)*5` | 0 | +1 | 5 | `[]` |
| hết | – | `return 5 + (+1)*0` | – | – | **5** | `[]` |

### 10.2 Ghép lại: mỗi loại ký tự làm gì?

| Ký tự | Việc làm | Vì sao |
|---|---|---|
| chữ số | `num = num * 10 + d` | đang ghép số nhiều chữ số |
| `'+'`/`'-'` | `result += sign * num`, `num = 0`, `sign = ±1` | số trước đó đã xong → chốt; dấu cho số tiếp theo |
| `'('` | `stack.append((result, sign))`, `result, sign = 0, 1` | đóng băng mức ngoài, mở một biểu thức mới |
| `')'` | `result += sign * num`, `num = 0`, `r, sg = stack.pop()`, `result = r + sg * result` | nhóm xong → giá trị nhóm là **một hạng tử** của mức ngoài, mang dấu `sg` |
| `' '` | không làm gì | |
| hết chuỗi | `return result + sign * num` | chốt số cuối |

In [14]:
def calculate_stack(s: str) -> int:
    """Cách 2 (chính): một lượt duyệt, stack lưu (result, sign) của mức ngoài. O(n)."""
    stack = []                             # mỗi phần tử: (result bên ngoài, dấu trước '(')
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == "+" or ch == "-":
            result += sign * num           # chốt số vừa đọc
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append((result, sign))   # đóng băng mức ngoài
            result, sign = 0, 1            # bắt đầu biểu thức mới bên trong
        elif ch == ")":
            result += sign * num           # chốt số cuối trong ngoặc
            num = 0
            prev_result, prev_sign = stack.pop()
            result = prev_result + prev_sign * result   # nhóm = 1 hạng tử của mức ngoài
        # ch == ' ': bỏ qua
    return result + sign * num             # chốt số cuối cùng


cases = examples + [("10 - (2 + 3)", 5), ("-(2 + 3)", -5), ("1-(     -2)", 3), ("- (3 + (4 + 5))", -12)]
for s, expected in cases:
    got = calculate_stack(s)
    print(f"{s!r:24} -> {got:4}   kỳ vọng {expected:4}   oracle {oracle(s):4}")
    assert got == expected == oracle(s)

'1 + 1'                  ->    2   kỳ vọng    2   oracle    2
' 2-1 + 2 '              ->    3   kỳ vọng    3   oracle    3
'(1+(4+5+2)-3)+(6+8)'    ->   23   kỳ vọng   23   oracle   23
'10 - (2 + 3)'           ->    5   kỳ vọng    5   oracle    5
'-(2 + 3)'               ->   -5   kỳ vọng   -5   oracle   -5
'1-(     -2)'            ->    3   kỳ vọng    3   oracle    3
'- (3 + (4 + 5))'        ->  -12   kỳ vọng  -12   oracle  -12


### 10.3 Trace đầy đủ Example 3: `"(1+(4+5+2)-3)+(6+8)"`

Chuỗi có 19 ký tự (không có khoảng trắng) nên bảng có 19 bước, mỗi bước một ký tự, ghi trạng thái **sau khi** xử lý ký tự đó; dòng cuối là lần chốt khi hết chuỗi. Trong cột `stack`, `1` nghĩa là dấu `+1`.

| Bước | Ký tự | Việc làm | `num` | `sign` | `result` | `stack` (sau bước) |
|:-:|:-:|---|:-:|:-:|:-:|---|
| 1 | `(` | `cất (0, +1) vào stack, reset` | 0 | +1 | 0 | `[(0, 1)]` |
| 2 | `1` | `ghép chữ số -> num = 1` | 1 | +1 | 0 | `[(0, 1)]` |
| 3 | `+` | `chốt +1 vào result, sign = +1` | 0 | +1 | 1 | `[(0, 1)]` |
| 4 | `(` | `cất (1, +1) vào stack, reset` | 0 | +1 | 0 | `[(0, 1), (1, 1)]` |
| 5 | `4` | `ghép chữ số -> num = 4` | 4 | +1 | 0 | `[(0, 1), (1, 1)]` |
| 6 | `+` | `chốt +4 vào result, sign = +1` | 0 | +1 | 4 | `[(0, 1), (1, 1)]` |
| 7 | `5` | `ghép chữ số -> num = 5` | 5 | +1 | 4 | `[(0, 1), (1, 1)]` |
| 8 | `+` | `chốt +5 vào result, sign = +1` | 0 | +1 | 9 | `[(0, 1), (1, 1)]` |
| 9 | `2` | `ghép chữ số -> num = 2` | 2 | +1 | 9 | `[(0, 1), (1, 1)]` |
| 10 | `)` | `nhóm = 11; lấy (1, +1) ra: 1 + (+1)*11` | 0 | +1 | 12 | `[(0, 1)]` |
| 11 | `-` | `chốt +0 vào result, sign = -1` | 0 | -1 | 12 | `[(0, 1)]` |
| 12 | `3` | `ghép chữ số -> num = 3` | 3 | -1 | 12 | `[(0, 1)]` |
| 13 | `)` | `nhóm = 9; lấy (0, +1) ra: 0 + (+1)*9` | 0 | -1 | 9 | `[]` |
| 14 | `+` | `chốt +0 vào result, sign = +1` | 0 | +1 | 9 | `[]` |
| 15 | `(` | `cất (9, +1) vào stack, reset` | 0 | +1 | 0 | `[(9, 1)]` |
| 16 | `6` | `ghép chữ số -> num = 6` | 6 | +1 | 0 | `[(9, 1)]` |
| 17 | `+` | `chốt +6 vào result, sign = +1` | 0 | +1 | 6 | `[(9, 1)]` |
| 18 | `8` | `ghép chữ số -> num = 8` | 8 | +1 | 6 | `[(9, 1)]` |
| 19 | `)` | `nhóm = 14; lấy (9, +1) ra: 9 + (+1)*14` | 0 | +1 | 23 | `[]` |
| hết | – | `return 23 + (+1)*0` | – | – | **23** | `[]` |

Những điểm đáng nhìn kỹ:

- **Bước 1** gặp `'('` ngay đầu: cất `(0, +1)` — "bên ngoài chưa có gì, nhóm này mang dấu cộng".
- **Bước 4** gặp `'('` lần hai: cất `(1, +1)` — mức ngoài (ngoặc thứ nhất) đã có `result = 1`.
- **Bước 10** gặp `')'`: nhóm trong cùng có giá trị `4 + 5 + 2 = 11`, ghép với `(1, +1)` → `result = 12`.
- **Bước 13**: nhóm thứ nhất `1 + 11 - 3 = 9`, ghép với `(0, +1)` → `9`. Stack rỗng trở lại.
- **Bước 19**: nhóm `(6+8) = 14` ghép với `(9, +1)` → `23`. Hết chuỗi, `num = 0` nên chốt cuối không đổi gì.

Cell dưới là `trace_stack` — cùng logic với `calculate_stack`, chỉ thêm `print`. Output phải khớp từng dòng với bảng trên.

In [15]:
def trace_stack(s: str) -> int:
    stack = []
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch == " ":
            continue                                 # khoảng trắng: bỏ qua, không in
        if ch.isdigit():
            num = num * 10 + int(ch)
            action = f"ghép chữ số -> num = {num}"
        elif ch in "+-":
            result += sign * num
            action = f"chốt {sign * num:+d} vào result, sign = {'+1' if ch == '+' else '-1'}"
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append((result, sign))
            action = f"cất ({result}, {sign:+d}) vào stack, reset"
            result, sign = 0, 1
        else:                                        # ')'
            result += sign * num
            num = 0
            r, sg = stack.pop()
            action = f"nhóm = {result}; lấy ({r}, {sg:+d}) ra: {r} + ({sg:+d})*{result}"
            result = r + sg * result
        print(f"{ch!r:4}| {action:44}| num={num:<2} sign={sign:+d} result={result:<3} stack={stack}")
    final = result + sign * num
    print(f"Hết chuỗi: return {result} + ({sign:+d})*{num} = {final}")
    return final

assert trace_stack("(1+(4+5+2)-3)+(6+8)") == calculate_stack("(1+(4+5+2)-3)+(6+8)") == 23

'(' | cất (0, +1) vào stack, reset                | num=0  sign=+1 result=0   stack=[(0, 1)]
'1' | ghép chữ số -> num = 1                      | num=1  sign=+1 result=0   stack=[(0, 1)]
'+' | chốt +1 vào result, sign = +1               | num=0  sign=+1 result=1   stack=[(0, 1)]
'(' | cất (1, +1) vào stack, reset                | num=0  sign=+1 result=0   stack=[(0, 1), (1, 1)]
'4' | ghép chữ số -> num = 4                      | num=4  sign=+1 result=0   stack=[(0, 1), (1, 1)]
'+' | chốt +4 vào result, sign = +1               | num=0  sign=+1 result=4   stack=[(0, 1), (1, 1)]
'5' | ghép chữ số -> num = 5                      | num=5  sign=+1 result=4   stack=[(0, 1), (1, 1)]
'+' | chốt +5 vào result, sign = +1               | num=0  sign=+1 result=9   stack=[(0, 1), (1, 1)]
'2' | ghép chữ số -> num = 2                      | num=2  sign=+1 result=9   stack=[(0, 1), (1, 1)]
')' | nhóm = 11; lấy (1, +1) ra: 1 + (+1)*11      | num=0  sign=+1 result=12  stack=[(0, 1)]
'-' | chốt +0 vào resu

### 10.4 Dấu trừ một ngôi được xử lý "miễn phí" — trace `"-(2+3)"` và `"1-(     -2)"`

Đây là chỗ nhiều bạn lo nhất: "phải viết `if` riêng cho dấu trừ một ngôi chứ?" — **Không cần.** Hãy xem điều gì xảy ra:

- `"-(2+3)"`: ký tự đầu là `'-'`. Ta chốt `num = 0` → `result += (+1) * 0 = 0` (vô hại), rồi `sign = -1`. Gặp `'('` → cất `(0, -1)`: "bên ngoài bằng 0, nhóm này mang dấu trừ". Khi `')'` đến, nhóm `= 5` → `result = 0 + (-1) * 5 = -5`. Đúng!
- `"1-(     -2)"`: sau `'('` ta đã reset `result = 0`, `sign = +1` — y hệt **đầu một biểu thức mới**. Dấu `'-'` sau đó lại chỉ "chốt số 0" rồi đặt `sign = -1`. Khoảng trắng ở giữa bị bỏ qua nên không ảnh hưởng.

Tóm lại: dấu `'-'` một ngôi luôn đứng ở **đầu** một biểu thức (đầu chuỗi hoặc ngay sau `'('`), mà ở đầu biểu thức thì `result = 0` và `num = 0` → `-x` được tính như `0 - x`. Đó là lý do cách này gọn đến vậy.

In [16]:
assert trace_stack("-(2+3)") == -5
print()
assert trace_stack("1-(     -2)") == 3

'-' | chốt +0 vào result, sign = -1               | num=0  sign=-1 result=0   stack=[]
'(' | cất (0, -1) vào stack, reset                | num=0  sign=+1 result=0   stack=[(0, -1)]
'2' | ghép chữ số -> num = 2                      | num=2  sign=+1 result=0   stack=[(0, -1)]
'+' | chốt +2 vào result, sign = +1               | num=0  sign=+1 result=2   stack=[(0, -1)]
'3' | ghép chữ số -> num = 3                      | num=3  sign=+1 result=2   stack=[(0, -1)]
')' | nhóm = 5; lấy (0, -1) ra: 0 + (-1)*5        | num=0  sign=+1 result=-5  stack=[]
Hết chuỗi: return -5 + (+1)*0 = -5

'1' | ghép chữ số -> num = 1                      | num=1  sign=+1 result=0   stack=[]
'-' | chốt +1 vào result, sign = -1               | num=0  sign=-1 result=1   stack=[]
'(' | cất (1, -1) vào stack, reset                | num=0  sign=+1 result=0   stack=[(1, -1)]
'-' | chốt +0 vào result, sign = -1               | num=0  sign=-1 result=0   stack=[(1, -1)]
'2' | ghép chữ số -> num = 2                      | 

### 10.5 Vì sao Cách 2 đúng? (invariant bằng lời)

Gọi **mức hiện tại** là cặp ngoặc trong cùng đang mở tại vị trí đang đọc (mức 0 là cả biểu thức). Sau khi xử lý **mỗi** ký tự, ba điều sau luôn đúng — đó là **bất biến** (invariant):

1. `result` = tổng có dấu của các hạng tử **đã đọc xong** trong mức hiện tại (chưa tính số đang ghép dở).
2. `num` = giá trị số đang ghép dở (hoặc 0), và `sign` = dấu sẽ gắn cho nó.
3. `stack` có đúng **một phần tử cho mỗi mức ngoặc bên ngoài** đang mở, xếp từ ngoài vào trong; phần tử `(r, sg)` nghĩa là: ở mức cha, các hạng tử trước `'('` có tổng `r`, và cả nhóm ngoặc này sẽ được nhân với `sg`.

Kiểm tra từng nhánh **giữ nguyên** bất biến:

- **Chữ số:** chỉ kéo dài `num` → (2) vẫn đúng.
- **`'+'`/`'-'`:** hạng tử `sign * num` vừa xong → cộng vào `result` (1); `num = 0`, `sign` mới (2).
- **`'('`:** một mức mới bắt đầu, chưa có hạng tử nào → `result = 0`, `sign = +1`; trạng thái mức cha được cất lên đỉnh stack (3).
- **`')'`:** chốt `num` → `result` là giá trị **cả nhóm**. Mà cả nhóm chính là **một hạng tử** của mức cha với dấu `sg` → `result` của mức cha mới là `r + sg * result` (1 cho mức cha). `num = 0`.
- **Hết chuỗi:** ngoặc cân bằng nên stack rỗng, ta đang ở mức 0 → đáp án là `result + sign * num`.

Theo quy nạp trên số ký tự đã đọc, kết quả trả về đúng bằng giá trị biểu thức.

**Một chi tiết tinh tế:** ngay sau `')'`, biến `sign` vẫn còn giữ dấu cũ của mức con (không còn ý nghĩa). Nhưng vô hại: `num = 0`, và ký tự có nghĩa kế tiếp chỉ có thể là toán tử (đặt lại `sign`), một `')'` khác (chốt `sign * 0 = 0`), hoặc hết chuỗi (`+ sign * 0`). Bạn có thể thấy điều này ở bước cuối của trace `"-(2+3)"`.

### Độ phức tạp

Gọi `n = len(s)`, `d` = độ sâu lồng ngoặc lớn nhất.

- **Time `O(n)`**: mỗi ký tự được xử lý đúng một lần, mỗi lần `O(1)` (push/pop stack là `O(1)`).
- **Space `O(d)`**: stack chứa tối đa `d` cặp; trường hợp xấu nhất `d ≈ n / 2` nên `O(n)`.

## 11. Cách 3 — Stack chỉ lưu dấu ("phân phối dấu vào ngoặc")

Nhìn lại phép bỏ ngoặc ở mục 7: `1 - (2 - (3 + 4)) = 1 - 2 + 3 + 4`. Mỗi số cuối cùng chỉ được **cộng hoặc trừ** vào kết quả, với **dấu thật** (effective sign) bằng:

```text
dấu thật của một số = (dấu ngay trước nó) x (tích các dấu đứng trước MỌI cặp ngoặc bao quanh nó)
```

| Số | Dấu ngay trước | Các ngoặc bao quanh mang dấu | Dấu thật |
|:-:|:-:|---|:-:|
| `1` | `+` | (không có) | `+` |
| `2` | `+` | `-( ... )` | `-` |
| `3` | `+` | `-( ... )`, rồi `-( ... )` | `+` |
| `4` | `+` | `-( ... )`, rồi `-( ... )` | `+` |

Vậy ta **không cần** cất `result` của mức ngoài — chỉ cần một biến `result` chung và một stack `ops` lưu **dấu hiệu lực của từng mức ngoặc**:

- `ops = [1]` ban đầu (mức 0 mang dấu `+`).
- `sign` bây giờ là **dấu thật** của số sắp tới: gặp `'+'` → `sign = ops[-1]`; gặp `'-'` → `sign = -ops[-1]`.
- Gặp `'('` → `ops.append(sign)`: dấu thật của cả nhóm trở thành dấu hiệu lực của mức mới.
- Gặp `')'` → chốt số, rồi `ops.pop()`.

Cell dưới in trạng thái sau mỗi ký tự của `"1-(2-(3+4))"` để thấy `ops` lớn lên và co lại theo ngoặc. Độ phức tạp giống Cách 2: **Time `O(n)`, Space `O(d)`**. Ưu điểm: chỉ một `result`. Nhược điểm: ý tưởng "nhân dấu" chỉ đẹp khi biểu thức **chỉ có `+`, `-`** — thêm `*`, `/` là hỏng (dấu không còn "phân phối" đơn giản được nữa).

In [17]:
def calculate_sign_stack(s: str) -> int:
    """Cách 3: mỗi số được cộng thẳng vào result với DẤU THẬT của nó. O(n)."""
    result, num, sign = 0, 0, 1
    ops = [1]                        # ops[-1] = dấu hiệu lực của mức ngoặc hiện tại
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = ops[-1] if ch == "+" else -ops[-1]
        elif ch == "(":
            ops.append(sign)         # dấu thật của cả nhóm
        elif ch == ")":
            result += sign * num
            num = 0
            ops.pop()
    return result + sign * num


def trace_sign_stack(s: str) -> int:
    result, num, sign, ops = 0, 0, 1, [1]
    for ch in s:
        if ch == " ":
            continue
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = ops[-1] if ch == "+" else -ops[-1]
        elif ch == "(":
            ops.append(sign)
        else:
            result += sign * num
            num = 0
            ops.pop()
        print(f"{ch!r:4} num={num:<2} sign(dấu thật)={sign:+d}  ops={str(ops):12} result={result}")
    print("Kết quả:", result + sign * num)
    return result + sign * num

assert trace_sign_stack("1-(2-(3+4))") == 6
for s, expected in cases:
    assert calculate_sign_stack(s) == expected
print("calculate_sign_stack đúng trên mọi case ở mục 10")

'1'  num=1  sign(dấu thật)=+1  ops=[1]          result=0
'-'  num=0  sign(dấu thật)=-1  ops=[1]          result=1
'('  num=0  sign(dấu thật)=-1  ops=[1, -1]      result=1
'2'  num=2  sign(dấu thật)=-1  ops=[1, -1]      result=1
'-'  num=0  sign(dấu thật)=+1  ops=[1, -1]      result=-1
'('  num=0  sign(dấu thật)=+1  ops=[1, -1, 1]   result=-1
'3'  num=3  sign(dấu thật)=+1  ops=[1, -1, 1]   result=-1
'+'  num=0  sign(dấu thật)=+1  ops=[1, -1, 1]   result=2
'4'  num=4  sign(dấu thật)=+1  ops=[1, -1, 1]   result=2
')'  num=0  sign(dấu thật)=+1  ops=[1, -1]      result=6
')'  num=0  sign(dấu thật)=+1  ops=[1]          result=6
Kết quả: 6
calculate_sign_stack đúng trên mọi case ở mục 10


## 12. Kiến thức nền 4 — Đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** để giải một bài toán con cùng dạng nhưng nhỏ hơn. Mỗi hàm đệ quy cần:

- **Trường hợp cơ sở** (base case): bài đủ nhỏ để trả lời ngay, không gọi tiếp.
- **Bước đệ quy**: chia bài thành phần nhỏ hơn và gọi lại chính mình.

Ví dụ rất gần với ngoặc lồng nhau: tính tổng một **list lồng nhau** `[1, [2, [3, 4]], 5]`. Mỗi `[` giống một `'('` — bên trong là một "bài toán con" cùng dạng:

```text
nested_sum([1, [2, [3, 4]], 5])
  = 1 + nested_sum([2, [3, 4]]) + 5
  = 1 + (2 + nested_sum([3, 4])) + 5
  = 1 + (2 + 7) + 5 = 15
```

Khi một hàm gọi hàm khác (hay gọi chính nó), Python cất trạng thái của lời gọi hiện tại (biến cục bộ, đang ở dòng nào) vào **call stack** (ngăn xếp lời gọi) rồi mới chạy lời gọi mới. Lời gọi mới trả về → lấy trạng thái cũ ra chạy tiếp. Nghĩa là **đệ quy chính là một stack do Python tự quản lý giúp ta** — đó là lý do Cách 2 (stack tự viết) và Cách 4 (đệ quy) thực chất là anh em.

Cell dưới in thụt lề theo độ sâu để bạn "nhìn thấy" call stack lớn lên rồi co lại.

In [18]:
def nested_sum(items: list, depth: int = 0) -> int:
    pad = "    " * depth
    print(f"{pad}vào nested_sum({items})")
    total = 0
    for x in items:
        if isinstance(x, list):
            total += nested_sum(x, depth + 1)     # gặp "ngoặc mở" -> gọi đệ quy
        else:
            total += x
    print(f"{pad}ra, trả về {total}")
    return total

assert nested_sum([1, [2, [3, 4]], 5]) == 15

vào nested_sum([1, [2, [3, 4]], 5])
    vào nested_sum([2, [3, 4]])
        vào nested_sum([3, 4])
        ra, trả về 7
    ra, trả về 9
ra, trả về 15


## 13. Cách 4 — Đệ quy (recursive descent): mỗi cặp ngoặc là một lời gọi hàm

**Recursive descent** (phân tích đệ quy đi xuống) là kỹ thuật viết bộ phân tích bằng các hàm đệ quy bám sát "ngữ pháp" của biểu thức. Với bài này, ngữ pháp nói bằng lời rất ngắn:

- Một **biểu thức** là dãy các **hạng tử** nối với nhau bằng `+` hoặc `-` (hạng tử đầu có thể mang dấu `-` một ngôi).
- Một **hạng tử** là một **số**, hoặc **`(` một biểu thức `)`**.

Câu thứ hai là chìa khóa: **cả cụm ngoặc đóng vai trò giống hệt một con số.** Vậy ta viết hàm `parse(i)`:

- Bắt đầu đọc từ vị trí `i`, tính một biểu thức bằng đúng bộ xương `eval_no_paren` (`result`, `sign`, `num`).
- Gặp `'('` → gọi đệ quy `parse(i + 1)` để tính cả nhóm, nhận về giá trị và **coi nó như `num`** (một con số vừa đọc xong).
- Gặp `')'` → nhóm hiện tại kết thúc: trả về giá trị.

**Hàm cần trả về 2 thứ:** giá trị **và** vị trí đã đọc tới (để lời gọi cha biết đọc tiếp từ đâu, không đọc lại các ký tự trong ngoặc). Python cho trả về tuple: `return value, i` và nhận bằng `num, i = parse(i + 1)`.

Quy ước: `parse` trả về `i` là **vị trí của `')'`** đã kết thúc nhóm; lời gọi cha sau đó chạy `i += 1` như mọi ký tự khác để bước qua `')'`.

Trace (cell thứ hai) in thụt lề theo **mức ngoặc**: mỗi lần vào ngoặc là một lời gọi mới, sâu hơn một nấc.

In [19]:
def calculate_recursive(s: str) -> int:
    """Cách 4: mỗi '(' gọi đệ quy parse; cả nhóm ngoặc được coi như một con số. O(n)."""
    n = len(s)

    def parse(i: int):
        """Tính biểu thức bắt đầu từ i, dừng ở ')' khớp hoặc hết chuỗi -> (giá trị, vị trí dừng)."""
        result, sign, num = 0, 1, 0
        while i < n:
            ch = s[i]
            if ch.isdigit():
                num = num * 10 + int(ch)
            elif ch in "+-":
                result += sign * num
                num = 0
                sign = 1 if ch == "+" else -1
            elif ch == "(":
                num, i = parse(i + 1)       # cả nhóm = một "con số"; i đang ở ')'
            elif ch == ")":
                return result + sign * num, i
            i += 1
        return result + sign * num, i

    value, _ = parse(0)
    return value

for s, expected in cases:
    assert calculate_recursive(s) == expected, s
print("calculate_recursive đúng trên mọi case ở mục 10")
print(calculate_recursive("(1+(4+5+2)-3)+(6+8)"))

calculate_recursive đúng trên mọi case ở mục 10
23


In [20]:
def trace_recursive(s: str) -> int:
    n = len(s)

    def parse(i: int, depth: int):
        pad = "    " * depth
        print(f"{pad}parse({i}) bắt đầu, mức ngoặc {depth}")
        result, sign, num = 0, 1, 0
        while i < n:
            ch = s[i]
            if ch.isdigit():
                num = num * 10 + int(ch)
            elif ch in "+-":
                result += sign * num
                num = 0
                sign = 1 if ch == "+" else -1
            elif ch == "(":
                num, i = parse(i + 1, depth + 1)
                print(f"{pad}  nhận giá trị nhóm = {num}, đọc tiếp sau vị trí {i}")
            elif ch == ")":
                print(f"{pad}gặp ')' ở vị trí {i} -> trả về {result + sign * num}")
                return result + sign * num, i
            i += 1
        print(f"{pad}hết chuỗi -> trả về {result + sign * num}")
        return result + sign * num, i

    return parse(0, 0)[0]

assert trace_recursive("(1+(4+5+2)-3)+(6+8)") == 23

parse(0) bắt đầu, mức ngoặc 0
    parse(1) bắt đầu, mức ngoặc 1
        parse(4) bắt đầu, mức ngoặc 2
        gặp ')' ở vị trí 9 -> trả về 11
      nhận giá trị nhóm = 11, đọc tiếp sau vị trí 9
    gặp ')' ở vị trí 12 -> trả về 9
  nhận giá trị nhóm = 9, đọc tiếp sau vị trí 12
    parse(15) bắt đầu, mức ngoặc 1
    gặp ')' ở vị trí 18 -> trả về 14
  nhận giá trị nhóm = 14, đọc tiếp sau vị trí 18
hết chuỗi -> trả về 23


### 13.1 Giới hạn độ sâu đệ quy của Python — vì sao Cách 4 không phải lời giải chính

Mỗi lời gọi đệ quy chiếm một "ô" trong call stack. Để tránh tràn bộ nhớ, Python đặt **giới hạn độ sâu** (mặc định `1000`, xem bằng `sys.getrecursionlimit()`); vượt quá sẽ báo `RecursionError`.

Với bài này, mỗi mức ngoặc lồng nhau là một lời gọi `parse`. Constraints cho phép chuỗi dài `3 * 10^5`, tức về lý thuyết có thể lồng sâu tới khoảng `1.5 * 10^5` mức — vượt xa `1000`.

- Có thể nới bằng `sys.setrecursionlimit(...)`, nhưng nới quá lớn có thể làm **sập hẳn** trình thông dịch (tràn stack của chính chương trình C bên dưới), không chỉ báo lỗi lịch sự.
- **Time `O(n)`** (mỗi ký tự đọc đúng một lần, chỉ số `i` chỉ tăng), **Space `O(d)`** cho call stack.

Kết luận: đệ quy là cách **dễ đọc, bám sát ngữ pháp, dễ mở rộng** (thêm `*`, `/` rất tự nhiên — xem Bài 6), nhưng với input lồng sâu thì **Cách 2 dùng stack tự quản lý an toàn hơn**. Cell dưới chứng minh bằng một chuỗi lồng sâu 3000 mức.

In [21]:
import sys

print("Giới hạn độ sâu đệ quy mặc định:", sys.getrecursionlimit())
deep = "(" * 3000 + "1" + ")" * 3000
print("len(deep) =", len(deep))

try:
    calculate_recursive(deep)
except RecursionError as e:
    print("Cách 4 (đệ quy):    ", type(e).__name__, "-", e)

print("Cách 2 (stack):     ", calculate_stack(deep))
print("Cách 3 (stack dấu): ", calculate_sign_stack(deep))

Giới hạn độ sâu đệ quy mặc định: 1000
len(deep) = 6001
Cách 4 (đệ quy):     RecursionError - maximum recursion depth exceeded
Cách 2 (stack):      1
Cách 3 (stack dấu):  1


## 14. Kiến thức nền 5 — Ký pháp hậu tố (RPN) và cách tính bằng stack

Cách ta viết `1 + 2` gọi là **trung tố** (infix): toán tử đứng **giữa** hai toán hạng. Nhà logic học Ba Lan Jan Łukasiewicz đề xuất cách viết khác; dạng "đảo ngược" của nó gọi là **ký pháp hậu tố** — *postfix* hay **RPN** (*Reverse Polish Notation*): toán tử đứng **sau** hai toán hạng.

| Trung tố (infix) | Hậu tố (RPN) |
|---|---|
| `1 + 2` | `1 2 +` |
| `(10 - 4) - 3` | `10 4 - 3 -` |
| `10 - (4 - 3)` | `10 4 3 - -` |

Điều kỳ diệu: **RPN không cần ngoặc** — thứ tự các toán tử đã nói lên ngoặc. Và tính RPN cực dễ bằng **một stack**:

```text
duyệt từng token:
    số        -> push
    toán tử   -> b = pop()  (toán hạng PHẢI — vào sau nên ra trước)
                 a = pop()  (toán hạng TRÁI)
                 push(a op b)
cuối cùng stack còn đúng 1 số: đáp án
```

Chú ý thứ tự `b` rồi `a`: với phép trừ, `a - b` khác `b - a`! (Đây chính là bài LeetCode #150 Evaluate Reverse Polish Notation.)

In [22]:
def eval_rpn(rpn: list) -> int:
    stack = []
    for tok in rpn:
        if isinstance(tok, int):
            stack.append(tok)
        else:
            b = stack.pop()        # toán hạng phải
            a = stack.pop()        # toán hạng trái
            stack.append(a + b if tok == "+" else a - b)
    return stack.pop()


def trace_rpn(rpn: list) -> int:
    stack = []
    for tok in rpn:
        if isinstance(tok, int):
            stack.append(tok)
            print(f"{tok!r:5} -> push                 stack = {stack}")
        else:
            b, a = stack.pop(), stack.pop()
            stack.append(a + b if tok == "+" else a - b)
            print(f"{tok!r:5} -> a={a}, b={b}: {a} {tok} {b} = {stack[-1]:<3} stack = {stack}")
    return stack.pop()

value = trace_rpn([10, 4, 3, "-", "-"])
assert value == eval_rpn([10, 4, 3, "-", "-"]) == 9
print("10 4 3 - -  =", value)
print("10 4 - 3 -  =", eval_rpn([10, 4, "-", 3, "-"]))

10    -> push                 stack = [10]
4     -> push                 stack = [10, 4]
3     -> push                 stack = [10, 4, 3]
'-'   -> a=4, b=3: 4 - 3 = 1   stack = [10, 1]
'-'   -> a=10, b=1: 10 - 1 = 9   stack = [9]
10 4 3 - -  = 9
10 4 - 3 -  = 3


## 15. Cách 5 — Shunting-yard: đổi trung tố sang RPN, rồi tính

**Shunting-yard** ("sân dồn toa tàu") là thuật toán của Edsger Dijkstra để đổi biểu thức trung tố sang RPN. Hình dung: các token chạy qua như toa tàu; **số** đi thẳng ra đường ray `output`, còn **toán tử** và `'('` phải tạm vào "đường tránh" là stack `ops`.

Quy tắc cho bài này (chỉ có `+`, `-` — cùng độ ưu tiên, kết hợp trái):

| Token | Việc làm |
|---|---|
| số | đưa thẳng ra `output` |
| `'('` | đẩy vào `ops` |
| `'+'`/`'-'` | đổ (pop) các toán tử trên đỉnh `ops` ra `output` cho tới khi gặp `'('` hoặc `ops` rỗng; rồi đẩy toán tử mới vào `ops` |
| `')'` | đổ `ops` ra `output` cho tới khi gặp `'('`; bỏ `'('` đi |
| hết token | đổ hết `ops` ra `output` |

**Vì sao toán tử mới phải "đổ" toán tử cũ cùng mức ra trước?** Vì kết hợp trái: `10 - 4 - 3` nghĩa là `(10 - 4) - 3`, nên dấu `-` thứ nhất phải được thực hiện **trước** dấu thứ hai → nó phải ra `output` trước: `10 4 - 3 -`.

### Dấu trừ một ngôi: mẹo "chèn số 0"

RPN ở trên chỉ biết toán tử **hai ngôi** (luôn pop 2 số). Với `"-(2+3)"`, dấu `'-'` không có toán hạng trái → `eval_rpn` sẽ pop từ stack rỗng và lỗi. Mẹo: theo mục 2, `'-'` một ngôi chỉ đứng **ở đầu** hoặc **ngay sau `'('`**, tức token đứng trước nó là "không có gì" hoặc `'('`. Khi gặp trường hợp đó, ta **đưa số `0` ra `output` trước**, biến `-x` thành `0 - x`.

**Vì sao chèn 0 an toàn ở bài này?** Biểu thức bắt đầu bằng `- a ± b ± c...` được hiểu (kết hợp trái) là `((0 - a) ± b) ± c...` = `(-a) ± b ± c...` — đúng nghĩa. Nếu bài có thêm `*`, `/` thì phải cẩn thận hơn (ví dụ coi dấu một ngôi là một toán tử riêng có độ ưu tiên cao nhất).

Vì `tokenize` đã bỏ hết khoảng trắng, "token đứng trước" ở đây không bao giờ là khoảng trắng → `"1-(     -2)"` cũng được nhận đúng là một ngôi.

**Độ phức tạp:** tokenize `O(n)` + shunting-yard `O(n)` (mỗi toán tử vào/ra `ops` đúng một lần) + tính RPN `O(n)` → **Time `O(n)`**; list token, `output` và các stack → **Space `O(n)`**. Ba lượt thay vì một, dài hơn Cách 2 — đổi lại đây là cách **tổng quát nhất**: thêm `*`, `/`, `^` chỉ cần thêm bảng độ ưu tiên.

In [23]:
def to_rpn(tokens: list) -> list:
    """Shunting-yard cho + - ( ) với mẹo chèn 0 cho dấu trừ một ngôi."""
    output, ops = [], []
    prev = None                                   # token đứng trước
    for tok in tokens:
        if isinstance(tok, int):
            output.append(tok)
        elif tok == "(":
            ops.append(tok)
        elif tok == ")":
            while ops[-1] != "(":
                output.append(ops.pop())
            ops.pop()                             # bỏ '('
        else:                                     # '+' hoặc '-'
            if tok == "-" and (prev is None or prev == "("):
                output.append(0)                  # một ngôi: -x  ->  0 - x
            while ops and ops[-1] != "(":
                output.append(ops.pop())
            ops.append(tok)
        prev = tok
    while ops:
        output.append(ops.pop())
    return output


def calculate_rpn(s: str) -> int:
    """Cách 5: tokenize -> shunting-yard -> tính RPN. O(n)."""
    return eval_rpn(to_rpn(tokenize(s)))

for s in ["10 - 4 - 3", "10 - (4 - 3)", "-(2 + 3)", "(1+(4+5+2)-3)+(6+8)"]:
    print(f"{s!r:24} -> RPN: {show(to_rpn(tokenize(s))):28} = {calculate_rpn(s)}")
for s, expected in cases:
    assert calculate_rpn(s) == expected, s
print("calculate_rpn đúng trên mọi case ở mục 10")

'10 - 4 - 3'             -> RPN: 10 4 - 3 -                   = 3
'10 - (4 - 3)'           -> RPN: 10 4 3 - -                   = 9
'-(2 + 3)'               -> RPN: 0 2 3 + -                    = -5
'(1+(4+5+2)-3)+(6+8)'    -> RPN: 1 4 5 + 2 + + 3 - 6 8 + +    = 23
calculate_rpn đúng trên mọi case ở mục 10


### 15.1 Trace shunting-yard với `"1 - (-2 + 3)"`

Ví dụ này có đủ: toán tử hai ngôi, ngoặc, dấu trừ **một ngôi ngay sau `'('`**, và một lần "đổ" toán tử cùng mức.

| Token | Việc làm | `output` (RPN đang xây) | `ops` |
|:-:|---|---|---|
| `1` | `số -> đưa thẳng ra output` | `1` | `[]` |
| `-` | `đẩy '-' vào ops` | `1` | `['-']` |
| `(` | `'(' -> đẩy vào ops` | `1` | `['-', '(']` |
| `-` | `một ngôi: chèn 0; đẩy '-' vào ops` | `1 0` | `['-', '(', '-']` |
| `2` | `số -> đưa thẳng ra output` | `1 0 2` | `['-', '(', '-']` |
| `+` | `đổ ['-']; đẩy '+' vào ops` | `1 0 2 -` | `['-', '(', '+']` |
| `3` | `số -> đưa thẳng ra output` | `1 0 2 - 3` | `['-', '(', '+']` |
| `)` | `đổ ['+'] ra output, bỏ '('` | `1 0 2 - 3 +` | `['-']` |
| `(hết)` | `đổ nốt ['-'] ra output` | `1 0 2 - 3 + -` | `[]` |

RPN thu được `1 0 2 - 3 + -` nghĩa là `1 - ((0 - 2) + 3) = 1 - 1 = 0`. Để ý dòng token `'+'`: dấu `'-'` (thật ra là `0 - 2`) nằm trên đỉnh `ops`, cùng mức ưu tiên nên bị đổ ra trước — đúng tinh thần kết hợp trái. Cell dưới in đúng bảng này rồi tính RPN.

In [24]:
def trace_shunting(s: str) -> list:
    output, ops, prev = [], [], None
    for tok in tokenize(s):
        if isinstance(tok, int):
            output.append(tok)
            action = "số -> đưa thẳng ra output"
        elif tok == "(":
            ops.append(tok)
            action = "'(' -> đẩy vào ops"
        elif tok == ")":
            moved = []
            while ops[-1] != "(":
                moved.append(ops.pop())
                output.append(moved[-1])
            ops.pop()
            action = f"đổ {moved} ra output, bỏ '('"
        else:
            action = ""
            if tok == "-" and (prev is None or prev == "("):
                output.append(0)
                action = "một ngôi: chèn 0; "
            moved = []
            while ops and ops[-1] != "(":
                moved.append(ops.pop())
                output.append(moved[-1])
            if moved:
                action += f"đổ {moved}; "
            ops.append(tok)
            action += f"đẩy '{tok}' vào ops"
        prev = tok
        print(f"{str(tok):>5} | {action:38} | output = {show(output):14} | ops = {ops}")
    moved = []
    while ops:
        moved.append(ops.pop())
        output.append(moved[-1])
    print(f"(hết) | {'đổ nốt ' + str(moved) + ' ra output':38} | output = {show(output):14} | ops = {ops}")
    return output

rpn = trace_shunting("1 - (-2 + 3)")
assert rpn == to_rpn(tokenize("1 - (-2 + 3)"))     # trace phải khớp to_rpn thật
print("Giá trị RPN:", eval_rpn(rpn), "  oracle:", oracle("1 - (-2 + 3)"))

    1 | số -> đưa thẳng ra output              | output = 1              | ops = []
    - | đẩy '-' vào ops                        | output = 1              | ops = ['-']
    ( | '(' -> đẩy vào ops                     | output = 1              | ops = ['-', '(']
    - | một ngôi: chèn 0; đẩy '-' vào ops      | output = 1 0            | ops = ['-', '(', '-']
    2 | số -> đưa thẳng ra output              | output = 1 0 2          | ops = ['-', '(', '-']
    + | đổ ['-']; đẩy '+' vào ops              | output = 1 0 2 -        | ops = ['-', '(', '+']
    3 | số -> đưa thẳng ra output              | output = 1 0 2 - 3      | ops = ['-', '(', '+']
    ) | đổ ['+'] ra output, bỏ '('             | output = 1 0 2 - 3 +    | ops = ['-']
(hết) | đổ nốt ['-'] ra output                 | output = 1 0 2 - 3 + -  | ops = []
Giá trị RPN: 0   oracle: 0


## 16. Code nộp LeetCode

Ta nộp **Cách 2** (stack `(result, sign)`): một lượt duyệt, `O(n)`, không đệ quy nên không sợ độ sâu, và rất ngắn. Chuyển hàm `calculate_stack` vào `class Solution` với đúng signature `def calculate(self, s: str) -> int`. Không cần `import` gì (khoảng trắng tự động bị bỏ qua vì không khớp nhánh nào).

In [25]:
class Solution:
    def calculate(self, s: str) -> int:
        stack = []                       # mỗi phần tử: (result của mức ngoài, dấu trước '(')
        result, sign, num = 0, 1, 0
        for ch in s:
            if ch.isdigit():
                num = num * 10 + int(ch)
            elif ch == "+" or ch == "-":
                result += sign * num
                num = 0
                sign = 1 if ch == "+" else -1
            elif ch == "(":
                stack.append((result, sign))
                result, sign = 0, 1
            elif ch == ")":
                result += sign * num
                num = 0
                prev_result, prev_sign = stack.pop()
                result = prev_result + prev_sign * result
        return result + sign * num

### 16.1 Bộ test: ví dụ đề + edge case

Mỗi test được kiểm tra **ba lần**: với giá trị kỳ vọng ghi tay, với `oracle` (`eval`), và với **cả 5 cách** đã học. Các nhóm edge case:

- **Ví dụ đề** (3 ca), gồm cả khoảng trắng đầu/cuối.
- **Chỉ một số**, có/không khoảng trắng: `"42"`, `"   7   "`.
- **Dấu trừ một ngôi**: ở đầu (`"-1"`), trước ngoặc (`"-(2 + 3)"`), ngay sau `'('` cách bởi khoảng trắng (`"1-(     -2)"`), lồng nhau (`"-(-(-1))"`).
- **Dấu trừ lồng sâu**: `"1 - (2 - (3 - (4 - 5)))"` — mỗi tầng lật dấu một lần.
- **Ngoặc bọc một số**: `"(((7)))"`.
- **Biên 32-bit**: `"2147483647"` (= `2^31 - 1`) và `"0-2147483647"`.

In [26]:
tests = [
    ("1 + 1", 2),                       # Example 1
    (" 2-1 + 2 ", 3),                   # Example 2: khoảng trắng đầu/cuối
    ("(1+(4+5+2)-3)+(6+8)", 23),        # Example 3: ngoặc lồng
    ("42", 42),                          # chỉ một số
    ("   7   ", 7),                      # một số + nhiều khoảng trắng
    ("-1", -1),                          # '-' một ngôi ở đầu
    ("-(2 + 3)", -5),                    # '-' một ngôi trước ngoặc
    ("- (3 + (4 + 5))", -12),            # '-' một ngôi, khoảng trắng phía sau
    ("1-(     -2)", 3),                  # '-' một ngôi ngay sau '(' (cách bởi khoảng trắng)
    ("-(-(-1))", -1),                    # một ngôi lồng nhau
    ("1 - (2 - (3 - (4 - 5)))", 3),      # dấu trừ lồng sâu
    ("(((7)))", 7),                      # ngoặc bọc một số
    ("10 - (2 + 3) - (4 - (1 + 1))", 3), # nhiều nhóm
    ("2147483647", 2147483647),          # 2^31 - 1
    ("0-2147483647", -2147483647),       # biên âm
]

sol = Solution()
methods = {
    "reduce": calculate_reduce,
    "stack": calculate_stack,
    "sign_stack": calculate_sign_stack,
    "recursive": calculate_recursive,
    "rpn": calculate_rpn,
}
for s, expected in tests:
    got = sol.calculate(s)
    assert got == expected == oracle(s), (s, got, expected)
    for name, f in methods.items():
        assert f(s) == expected, (name, s)
    print(f"{s!r:32} -> {got:11}  PASS")
print("Tất cả", len(tests), "test PASS cho Solution, oracle và cả 5 cách!")

'1 + 1'                          ->           2  PASS
' 2-1 + 2 '                      ->           3  PASS
'(1+(4+5+2)-3)+(6+8)'            ->          23  PASS
'42'                             ->          42  PASS
'   7   '                        ->           7  PASS
'-1'                             ->          -1  PASS
'-(2 + 3)'                       ->          -5  PASS
'- (3 + (4 + 5))'                ->         -12  PASS
'1-(     -2)'                    ->           3  PASS
'-(-(-1))'                       ->          -1  PASS
'1 - (2 - (3 - (4 - 5)))'        ->           3  PASS
'(((7)))'                        ->           7  PASS
'10 - (2 + 3) - (4 - (1 + 1))'   ->           3  PASS
'2147483647'                     ->  2147483647  PASS
'0-2147483647'                   -> -2147483647  PASS
Tất cả 15 test PASS cho Solution, oracle và cả 5 cách!


## 17. Đối chiếu chéo trên biểu thức ngẫu nhiên (có `random.seed`)

Test tay chỉ bao được những trường hợp ta **nghĩ ra**. Để bắt lỗi ở những trường hợp ta **không nghĩ ra**, ta sinh ngẫu nhiên hàng nghìn biểu thức hợp lệ và so mọi cách với `oracle`.

**Bộ sinh phải tôn trọng đúng constraints của đề** (nếu không, ta đang test một bài khác):

- Biểu thức = 1 tới 4 hạng tử nối bằng `+`/`-`; hạng tử là số `0..50` (không có chữ số 0 ở đầu, nên `eval` cũng hiểu được) hoặc một biểu thức con trong ngoặc (lồng tối đa 3 mức).
- `'-'` một ngôi chỉ được thêm vào **hạng tử đầu tiên** của một biểu thức — tức ở đầu chuỗi hoặc ngay sau `'('`. Như vậy không bao giờ có `'+'` một ngôi hay hai toán tử liền nhau.
- Khoảng trắng ngẫu nhiên **giữa** các token (không bao giờ chen vào giữa một số).

Hàm `respects_constraints` kiểm tra lại các điều kiện trên bằng list token (phòng khi bộ sinh có lỗi). `random.seed(224)` cố định "hạt giống" ngẫu nhiên → mỗi lần chạy notebook sinh **đúng cùng** một dãy biểu thức, kết quả lặp lại được.

In [27]:
import random

def rand_spaces() -> str:
    return " " * random.choice([0, 0, 0, 1, 1, 2])

def rand_term(depth: int) -> str:
    if depth < 3 and random.random() < 0.35:
        return "(" + rand_spaces() + rand_expr(depth + 1) + rand_spaces() + ")"
    return str(random.randint(0, 50))

def rand_expr(depth: int = 0) -> str:
    out = ""
    for j in range(random.randint(1, 4)):
        if j > 0:
            out += rand_spaces() + random.choice("+-") + rand_spaces()
        term = rand_term(depth)
        if j == 0 and random.random() < 0.3:       # '-' một ngôi chỉ ở đầu biểu thức
            term = "-" + rand_spaces() + term
        out += term
    return out

def respects_constraints(s: str) -> bool:
    toks = tokenize(s)
    for prev, cur in zip([None] + toks, toks):     # zip ghép từng cặp (token trước, token hiện tại)
        if cur in ("+", "-") and prev in ("+", "-"):
            return False                            # hai toán tử liền nhau
        if cur == "+" and (prev is None or prev == "("):
            return False                            # '+' một ngôi
    return True

random.seed(224)
N = 3000
for k in range(N):
    s = rand_spaces() + rand_expr() + rand_spaces()
    assert respects_constraints(s), s
    expected = oracle(s)
    assert -2**31 <= expected < 2**31
    if k < 4:
        print(f"ví dụ {k}: {s!r} = {expected}")
    assert sol.calculate(s) == expected, s
    for name, f in methods.items():
        assert f(s) == expected, (name, s, f(s), expected)
print(f"PASS: Solution + {len(methods)} cách x {N} biểu thức ngẫu nhiên đều khớp eval")

ví dụ 0: '34-  1' = 33
ví dụ 1: '( (-45  - 39 -  3+ 5) -28+  49)  -( 46) ' = -107
ví dụ 2: '35' = 35
ví dụ 3: '16 +21-  27  ' = 10


PASS: Solution + 5 cách x 3000 biểu thức ngẫu nhiên đều khớp eval


In [28]:
import time

big_flat = "1-" * 149_999 + "1"                    # 299 999 ký tự, 150 000 số
big_deep = "(" * 149_999 + "7" + ")" * 149_999     # lồng sâu 149 999 mức

for name, s in [("big_flat", big_flat), ("big_deep", big_deep)]:
    start = time.perf_counter()
    value = sol.calculate(s)
    elapsed = time.perf_counter() - start
    print(f"{name}: len = {len(s)}, kết quả = {value}, chạy dưới 1 giây? {elapsed < 1.0}")

print("Ước lượng công việc Cách 1 trên big_deep ≈ k² =", f"{149_999**2:.2e}", "-> không dám chạy!")

big_flat: len = 299999, kết quả = -149998, chạy dưới 1 giây? True


big_deep: len = 299999, kết quả = 7, chạy dưới 1 giây? True
Ước lượng công việc Cách 1 trên big_deep ≈ k² = 2.25e+10 -> không dám chạy!


## 18. So sánh các cách

`n = len(s)`, `d` = độ sâu lồng ngoặc lớn nhất (`d ≤ n / 2`).

| Cách | Ý tưởng | Time | Space | Ưu điểm | Nhược điểm |
|---|---|:-:|:-:|---|---|
| 0. `eval` | nhờ Python tính | `O(n)` | `O(n)` | 1 dòng; làm **trọng tài** test | **bị cấm**, nguy hiểm bảo mật, ngữ pháp khác đề |
| 1. Khử ngoặc trong cùng | như tính tay, lặp lại | `O(n²)` | `O(n)` | dễ nghĩ ra nhất | quá chậm khi `n = 3 * 10^5` |
| 2. Stack `(result, sign)` | đóng băng mức ngoài khi gặp `'('` | `O(n)` | `O(d)` | **một lượt**, ngắn, không đệ quy | phải nhớ đúng thứ tự cất/lấy |
| 3. Stack dấu | phân phối dấu vào ngoặc | `O(n)` | `O(d)` | chỉ một `result`, rất gọn | chỉ hợp khi **chỉ có** `+`, `-` |
| 4. Đệ quy | mỗi ngoặc là một lời gọi `parse` | `O(n)` | `O(d)` | bám sát ngữ pháp, dễ mở rộng | `RecursionError` khi lồng sâu hơn ~1000 |
| 5. Shunting-yard + RPN | đổi sang hậu tố rồi tính | `O(n)` | `O(n)` | **tổng quát nhất** (thêm `*`, `/`, `^` chỉ cần bảng ưu tiên) | dài, 3 lượt, dấu một ngôi cần mẹo |

**Nên nhớ:** đi phỏng vấn, viết **Cách 2** (hoặc Cách 3) trước; nếu được hỏi "thêm `*` `/` thì sao?" → nói tới Cách 4/Cách 5 (xem Bài 5, Bài 6).

## 19. Lỗi thường gặp

### Lỗi 1: Quên "chốt" số cuối cùng

```python
return result            # SAI
return result + sign * num   # ĐÚNG
```

Số cuối cùng không có toán tử nào phía sau để kích hoạt việc chốt, nên nó vẫn nằm trong `num`. Với `"1 + 1"`, bản sai trả về `1`. Lỗi này **không lộ ra** với biểu thức kết thúc bằng `')'` (vì `')'` đã chốt giúp) — ví dụ Example 3 vẫn đúng! Đó là lý do bộ test phải có ca kết thúc bằng một số.

In [29]:
def wrong_forget_final_flush(s: str) -> int:
    stack = []
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append((result, sign))
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num
            num = 0
            r, sg = stack.pop()
            result = r + sg * result
    return result                         # SAI: quên + sign * num

for s in ["1 + 1", "(1+(4+5+2)-3)+(6+8)"]:
    print(f"{s!r:24} bản sai = {wrong_forget_final_flush(s):3}   đúng = {calculate_stack(s)}")

'1 + 1'                  bản sai =   1   đúng = 2
'(1+(4+5+2)-3)+(6+8)'    bản sai =  23   đúng = 23


### Lỗi 2: Chỉ đọc một chữ số

```python
num = int(ch)                 # SAI: ghi đè, "12" thành 2
num = num * 10 + int(ch)      # ĐÚNG: ghép tiếp chữ số
```

Với `"12+3"`, bản sai đọc `'1'` rồi bị `'2'` ghi đè → ra `2 + 3 = 5` thay vì `15`. Ví dụ của đề toàn số một chữ số nên **không bắt được lỗi này** — luôn tự thêm test có số nhiều chữ số.

### Lỗi 3: Quên đặt lại `num = 0` sau `')'`

Sau khi chốt `num` ở `')'`, nếu không đặt lại `num = 0` thì lần chốt tiếp theo (ở toán tử kế tiếp) sẽ **cộng số đó thêm một lần nữa**. Với `"(1)+2"`: số `1` bị cộng 2 lần → ra `4` thay vì `3`.

In [30]:
def wrong_single_digit(s: str) -> int:
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = int(ch)                  # SAI: ghi đè thay vì ghép
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = 1 if ch == "+" else -1
    return result + sign * num


def wrong_no_reset_after_close(s: str) -> int:
    stack = []
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append((result, sign))
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num           # SAI: chốt xong nhưng quên num = 0
            r, sg = stack.pop()
            result = r + sg * result
    return result + sign * num

print("Lỗi 2: '12+3' ->", wrong_single_digit("12+3"), "  đúng =", calculate_stack("12+3"))
print("Lỗi 3: '(1)+2' ->", wrong_no_reset_after_close("(1)+2"), "  đúng =", calculate_stack("(1)+2"))

Lỗi 2: '12+3' -> 5   đúng = 15
Lỗi 3: '(1)+2' -> 4   đúng = 3


### Lỗi 4: Cất/lấy stack sai thứ tự

Nhiều lời giải cất **hai phần tử rời** thay vì một tuple: `stack.append(result); stack.append(sign)`. Khi lấy ra phải lấy **ngược lại** (LIFO): `sign` ra trước, `result` ra sau. Viết nhầm thứ tự sẽ nhân `result` của mức ngoài với giá trị nhóm! Với `"10-(2+3)"`: bản sai tính `-1 + 10 * 5 = 49`. Dùng **tuple** `(result, sign)` như notebook này loại bỏ hẳn lỗi này.

### Lỗi 5: RPN lấy toán hạng sai thứ tự

Trong `eval_rpn`, phần tử pop ra **trước** là toán hạng **phải** `b`. Viết `a = stack.pop(); b = stack.pop()` thì `10 - 4` thành `4 - 10 = -6`. Phép cộng không lộ lỗi (giao hoán), chỉ phép trừ mới lộ.

### Lỗi 6: Shunting-yard quên mẹo chèn 0 cho dấu trừ một ngôi

`"-(2+3)"` sẽ thành RPN `2 3 + -`: tới `'-'` thì stack chỉ còn một số `5`, pop lần hai từ stack rỗng → `IndexError`.

Một biến thể của lỗi này: nhận diện dấu một ngôi bằng cách nhìn **ký tự ngay trước** trong chuỗi gốc (`s[i-1] == '('`). Với `"1-(     -2)"`, ký tự ngay trước `'-'` là **khoảng trắng** → nhận diện sai. Phải nhìn **ký tự khác khoảng trắng** gần nhất (như `minus_kinds` ở mục 2) hoặc làm trên list token.

In [31]:
def wrong_flat_stack_order(s: str) -> int:
    stack = []
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append(result)          # cất 2 phần tử rời: result rồi sign
            stack.append(sign)
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num
            num = 0
            prev_result = stack.pop()     # SAI: phần tử ra trước là sign!
            prev_sign = stack.pop()
            result = prev_result + prev_sign * result
    return result + sign * num


def wrong_eval_rpn(rpn: list) -> int:
    stack = []
    for tok in rpn:
        if isinstance(tok, int):
            stack.append(tok)
        else:
            a = stack.pop()               # SAI: pop đầu tiên là toán hạng PHẢI
            b = stack.pop()
            stack.append(a + b if tok == "+" else a - b)
    return stack.pop()

print("Lỗi 4: '10-(2+3)' ->", wrong_flat_stack_order("10-(2+3)"), "  đúng =", calculate_stack("10-(2+3)"))
print("Lỗi 5: '10 4 -'   ->", wrong_eval_rpn([10, 4, "-"]), "  đúng =", eval_rpn([10, 4, "-"]))
try:
    eval_rpn([2, 3, "+", "-"])            # RPN của "-(2+3)" khi QUÊN chèn 0
except IndexError as e:
    print("Lỗi 6: RPN '2 3 + -' ->", type(e).__name__, "-", e)

Lỗi 4: '10-(2+3)' -> 49   đúng = 5
Lỗi 5: '10 4 -'   -> -6   đúng = 6
Lỗi 6: RPN '2 3 + -' -> IndexError - pop from empty list


### Lỗi 7, 8, 9 — không cần code

- **Lỗi 7: Nộp bằng `eval`.** Có thể chạy ra đúng đáp án, nhưng vi phạm thẳng yêu cầu đề; trong phỏng vấn là trượt. `eval` chỉ để làm trọng tài khi test.
- **Lỗi 8: Viết `if` riêng rườm rà cho dấu trừ một ngôi trong Cách 2.** Không cần — mục 10.4 đã cho thấy `-x` tự động được tính như `0 - x`. Thêm `if` dư chỉ tạo thêm chỗ để sai (đặc biệt khi có khoảng trắng).
- **Lỗi 9: Dùng đệ quy rồi tăng `sys.setrecursionlimit` thật lớn cho "chắc".** Với input lồng rất sâu, chương trình có thể bị sập hẳn (tràn stack C) thay vì báo lỗi. Muốn chắc chắn thì dùng stack tự quản lý (Cách 2/3).

## 20. Mẹo nhận dạng pattern

- **"Tính giá trị biểu thức dạng chuỗi"** → nghĩ ngay tới: tokenize/ghép số + **stack** hoặc **đệ quy**.
- **Thấy cấu trúc lồng nhau** (ngoặc `( )`, `[ ]`, thẻ HTML, list lồng list) → mẫu chung: **khi mở: cất trạng thái hiện tại vào stack rồi bắt đầu mới; khi đóng: lấy trạng thái cũ ra và ghép kết quả vào.** Cách 2 chính là mẫu này với trạng thái `(result, sign)`.
- **Chỉ có `+` và `-`** → mẹo "trừ là cộng số đối": biểu thức = tổng các hạng tử có dấu, chỉ cần biến `sign` (`+1`/`-1`), không cần độ ưu tiên.
- **Có thêm `*`, `/`** → cần độ ưu tiên: stack các hạng tử (#227), hoặc recursive descent theo tầng, hoặc shunting-yard với bảng ưu tiên.
- **Số nhiều chữ số** → `num = num * 10 + d`; **chốt** khi gặp toán tử, khi gặp `')'`, và **ở cuối chuỗi**.
- **Dấu trừ một ngôi** → nó chỉ ở đầu một biểu thức (đầu chuỗi hoặc sau `'('`): coi `-x` là `0 - x`.
- **`n` tới `3 * 10^5`** → cần một lượt `O(n)`; nếu dùng đệ quy, hỏi ngay "độ sâu tối đa là bao nhiêu?".

## 21. Biến thể và bài liên quan

| Bài | Độ khó | Liên hệ |
|---|---|---|
| #227 Basic Calculator II | Medium | `+ - * /`, **không** ngoặc → stack các hạng tử (Bài 5 bên dưới) |
| #772 Basic Calculator III | Hard (Premium) | `+ - * /` **và** ngoặc → gộp #224 + #227 (Bài 6) |
| #770 Basic Calculator IV | Hard | có cả **biến**, kết quả là đa thức |
| #150 Evaluate Reverse Polish Notation | Medium | đúng hàm `eval_rpn` (có thêm `*`, `/`) |
| #20 Valid Parentheses | Easy | stack ngoặc cơ bản |
| #394 Decode String | Medium | `3[a2[c]]`: khi gặp `'['` cất `(chuỗi đang xây, số lần lặp)` — y hệt mẫu "đóng băng" của Cách 2 |
| #726 Number of Atoms | Hard | `Mg(OH)2`: stack các bảng đếm, ngoặc đóng thì nhân hệ số |
| #1106 Parsing A Boolean Expression | Hard | recursive descent / stack trên biểu thức logic |
| #1597 Build Binary Expression Tree From Infix Expression | Hard (Premium) | shunting-yard nhưng dựng **cây** thay vì RPN |
| #282 Expression Add Operators | Hard | ngược lại: **chèn** toán tử vào dãy số (backtracking) |

### Biến thể nhỏ: viết lại biểu thức **không còn ngoặc**

Dùng đúng ý tưởng Cách 3 (dấu thật = tích các dấu bao ngoài), ta có thể "bỏ ngoặc" một biểu thức: `"1-(2-(3+4))"` → `"1-2+3+4"`. Thay vì cộng vào `result`, ta ghi lại từng hạng tử kèm dấu thật. Cell dưới kiểm chứng rằng biểu thức mới có cùng giá trị trên 500 biểu thức ngẫu nhiên.

In [32]:
def remove_parentheses(s: str) -> str:
    terms = []                        # các hạng tử kèm dấu thật, ví dụ ['+1', '-2', '+3', '+4']
    ops, sign, digits = [1], 1, ""
    for ch in s:
        if ch.isdigit():
            digits += ch
            continue
        if ch in "+-)" and digits:    # chốt số đang đọc
            terms.append(("+" if sign == 1 else "-") + digits)
            digits = ""
        if ch == "+":
            sign = ops[-1]
        elif ch == "-":
            sign = -ops[-1]
        elif ch == "(":
            ops.append(sign)
        elif ch == ")":
            ops.pop()
    if digits:
        terms.append(("+" if sign == 1 else "-") + digits)
    text = "".join(terms)
    return text[1:] if text.startswith("+") else text

for s in ["1-(2-(3+4))", "-(2 + 3)", "(1+(4+5+2)-3)+(6+8)", "1-(     -2)"]:
    print(f"{s!r:24} -> {remove_parentheses(s)!r}")

random.seed(7)
for _ in range(500):
    s = rand_expr()
    assert calculate_stack(remove_parentheses(s)) == oracle(s), s
print("remove_parentheses giữ nguyên giá trị trên 500 biểu thức ngẫu nhiên")

'1-(2-(3+4))'            -> '1-2+3+4'
'-(2 + 3)'               -> '-2-3'
'(1+(4+5+2)-3)+(6+8)'    -> '1+4+5+2-3+6+8'
'1-(     -2)'            -> '1+2'
remove_parentheses giữ nguyên giá trị trên 500 biểu thức ngẫu nhiên


## 22. Bài tập tự luyện

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 23) và **Đáp án** (mục 24, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

Các lệnh sau in ra gì? (Dùng đúng các hàm đã viết trong notebook.)

```python
tokenize("  7-(10 )")
eval_no_paren("8-3-2")
calculate_stack("-(-(4))")
to_rpn(tokenize("5-(1+2)"))
eval_no_paren("1-(2+3)")        # hàm này KHÔNG hiểu ngoặc
```

### Bài 2 — Tính tay trạng thái stack

**(a)** Chạy tay `calculate_stack("2-(5-(1+2))")`. Ghi `stack` và `result` **ngay sau** khi xử lý mỗi `'('` và mỗi `')'` (có 4 thời điểm như vậy), và kết quả cuối.

**(b)** Đổi `"3 - (4 - 5)"` sang RPN bằng shunting-yard, rồi tính RPN đó bằng stack. Nếu lỡ viết `eval_rpn` theo Lỗi 5 (lấy sai thứ tự toán hạng), kết quả sẽ là bao nhiêu?

### Bài 3 — Điền TODO: lời giải chính (Cách 2)

Hoàn thiện 5 chỗ TODO theo đúng bảng ở mục 10.2. Đừng nhìn lại code ở mục 10 nhé!

In [33]:
def exercise_calculate(s: str) -> int:
    stack = []
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            # TODO 1: ghép chữ số ch vào num
            pass
        elif ch in "+-":
            # TODO 2: chốt num vào result, đặt lại num, cập nhật sign
            pass
        elif ch == "(":
            # TODO 3: cất (result, sign) vào stack, rồi reset result và sign
            pass
        elif ch == ")":
            # TODO 4: chốt num, đặt lại num, lấy (prev_result, prev_sign) ra và ghép
            pass
    # TODO 5: đừng quên chốt số cuối cùng
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_calculate("1 + 1") == 2
# assert exercise_calculate(" 2-1 + 2 ") == 3
# assert exercise_calculate("(1+(4+5+2)-3)+(6+8)") == 23
# assert exercise_calculate("1-(     -2)") == 3
# assert exercise_calculate("12+3") == 15

### Bài 4 — Tự viết lại: Cách 4 (đệ quy)

Viết lại `calculate_recursive` **từ đầu** mà không nhìn mục 13. Yêu cầu: hàm con `parse(i)` trả về **tuple** `(giá trị, vị trí dừng)`; gặp `'('` thì gọi đệ quy và coi kết quả như một con số; gặp `')'` thì trả về.

In [34]:
def exercise_recursive(s: str) -> int:
    # TODO: viết hàm con parse(i) -> (value, i) và gọi parse(0)
    raise NotImplementedError

# Gợi ý test:
# assert exercise_recursive("(1+(4+5+2)-3)+(6+8)") == 23
# assert exercise_recursive("-(2 + 3)") == -5
# assert exercise_recursive("1 - (2 - (3 - (4 - 5)))") == 3
# assert exercise_recursive("42") == 42

### Bài 5 — Biến thể: LeetCode #227 Basic Calculator II

Biểu thức gồm **số nguyên không âm**, các toán tử `'+'`, `'-'`, `'*'`, `'/'` và khoảng trắng; **không có ngoặc**, không có dấu một ngôi. Phép chia là chia nguyên **làm tròn về 0** (truncate toward zero). Biểu thức luôn hợp lệ, `1 <= s.length <= 3 * 10^5`.

```text
"3+2*2"      -> 7
" 3/2 "      -> 1
" 3+5 / 2 "  -> 5
```

Bây giờ `*`, `/` được ưu tiên hơn `+`, `-`, nên mẹo "chỉ cần `sign`" không đủ nữa. Gợi ý ở mục 23.

**Cẩn thận:** trong Python, `//` làm tròn **xuống** (về âm vô cùng): `-3 // 2 == -2`, trong khi "làm tròn về 0" phải ra `-1`.

In [35]:
def calculate_ii(s: str) -> int:
    # TODO: stack các hạng tử + biến op = toán tử đứng TRƯỚC số đang đọc
    raise NotImplementedError

# Gợi ý test:
# assert calculate_ii("3+2*2") == 7
# assert calculate_ii(" 3/2 ") == 1
# assert calculate_ii(" 3+5 / 2 ") == 5
# assert calculate_ii("14-3/2") == 13          # bẫy làm tròn về 0
# assert calculate_ii("2*3*4-10/3") == 21

### Bài 6 — Mini challenge: `+ - * /` **và** ngoặc (phỏng theo #772 Basic Calculator III, Premium)

Gộp Bài 5 với bài chính. Giả định của bài tập này (do ta tự đặt, không phải đề gốc #772): số nguyên không âm, `+ - * /`, ngoặc, khoảng trắng; chia nguyên làm tròn về 0; **cho phép thêm** dấu `'-'` một ngôi giống bài 224 (ở đầu biểu thức hoặc ngay sau `'('`); biểu thức luôn hợp lệ và không chia cho 0.

```text
"2*(5+5*2)/3+(6/2+8)"         -> 21
"(2+6*3+5-(3*14/7+2)*5)+3"    -> -12
"-(3*2)+10"                    -> 4
"2*(-3)"                       -> -6
```

In [36]:
def calculate_iii(s: str) -> int:
    # TODO: recursive descent: parse(i) dùng stack hạng tử như Bài 5,
    #       gặp '(' thì gọi đệ quy và coi kết quả như một con số
    raise NotImplementedError

# Gợi ý test:
# assert calculate_iii("2*(5+5*2)/3+(6/2+8)") == 21
# assert calculate_iii("(2+6*3+5-(3*14/7+2)*5)+3") == -12
# assert calculate_iii("-(3*2)+10") == 4
# assert calculate_iii("2*(-3)") == -6
# assert calculate_iii("(1-6)/4") == -1        # làm tròn về 0, không phải -2

## 23. Gợi ý theo mức

**Bài 1**
- Mức 1: chạy tay từng hàm; nhớ `tokenize` bỏ khoảng trắng, còn `eval_no_paren` bỏ qua cả ngoặc.
- Mức 2: `"8-3-2"` là `(8-3)-2`; `"-(-(4))"` có hai lần lật dấu.
- Mức 3: `eval_no_paren("1-(2+3)")` thực chất tính `1-2+3`.

**Bài 2**
- Mức 1: chỉ có 2 lần push (ở 2 dấu `'('`) và 2 lần pop (ở 2 dấu `')'`).
- Mức 2: lần push thứ nhất cất `(2, -1)`; lần push thứ hai cất `(5, -1)`.
- Mức 3: ở `')'` thứ nhất, nhóm `(1+2) = 3` ghép với `(5, -1)` → `5 - 3 = 2`. Phần (b): `3 4 5 - -`.

**Bài 3**
- Mức 1: 5 chỗ trống ứng đúng 5 dòng của bảng mục 10.2.
- Mức 2: TODO 3 là `stack.append((result, sign))` rồi `result, sign = 0, 1`.
- Mức 3: TODO 4 là `result += sign * num; num = 0; r, sg = stack.pop(); result = r + sg * result`; TODO 5 là `return result + sign * num`.

**Bài 4**
- Mức 1: `parse` là `eval_no_paren` + 2 nhánh `'('` và `')'`, nhưng duyệt bằng `while i < n` (vì `i` phải nhảy qua cả nhóm ngoặc).
- Mức 2: nhánh `'('`: `num, i = parse(i + 1)`; nhánh `')'`: `return result + sign * num, i`.
- Mức 3: cuối thân vòng lặp luôn `i += 1` (bước qua cả `')'` mà lời gọi con dừng ở đó); hết chuỗi thì `return result + sign * num, i`.

**Bài 5**
- Mức 1: tách biểu thức thành các **hạng tử cộng** (`3 + 2*2` có hạng tử `3` và `4`); `*`, `/` chỉ tác động lên **hạng tử cuối cùng**.
- Mức 2: giữ biến `op` = toán tử đứng **trước** số đang đọc (ban đầu `'+'`). Khi một số đọc xong (gặp toán tử mới hoặc hết chuỗi): `'+'` → push `num`; `'-'` → push `-num`; `'*'` → `push(pop() * num)`; `'/'` → `push(chia_về_0(pop(), num))`. Đáp án `sum(stack)`.
- Mức 3: chia về 0 an toàn bằng số nguyên: `q = abs(a) // abs(b)`, nếu `a` và `b` trái dấu thì `-q`. Vì hạng tử bị push dưới dạng `-num` nên `a` có thể âm — đó là bẫy `"14-3/2"`.

**Bài 6**
- Mức 1: viết `parse(i)` như Bài 4, nhưng bên trong dùng kỹ thuật stack hạng tử của Bài 5.
- Mức 2: gặp `'('` → `num, i = parse(i + 1)` (cả nhóm là một số); gặp `')'` → áp `op` cuối với `num` rồi `return sum(stack), i`.
- Mức 3: dấu `'-'` một ngôi ở đầu tự động hoạt động: lúc đó `num = 0`, `op = '+'` → push `0`, rồi `op = '-'`, tức `0 - ...`. Viết một hàm phụ `apply(stack, op, num)` để không lặp code ở 3 chỗ chốt.

## 24. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:** `[7, '-', '(', 10, ')']`, `3`, `4`, `[5, 1, 2, '+', '-']`, `2`.

**Bài 2 (a):** `s = "2-(5-(1+2))"`

| Thời điểm | `stack` | `result` | Giải thích |
|---|---|:-:|---|
| sau `'('` thứ nhất (vị trí 2) | `[(2, -1)]` | 0 | cất "bên ngoài = 2, nhóm mang dấu trừ" |
| sau `'('` thứ hai (vị trí 5) | `[(2, -1), (5, -1)]` | 0 | cất "bên ngoài = 5, nhóm mang dấu trừ" |
| sau `')'` thứ nhất (vị trí 9) | `[(2, -1)]` | 2 | nhóm `1+2 = 3`; `5 + (-1)*3 = 2` |
| sau `')'` thứ hai (vị trí 10) | `[]` | 0 | nhóm `= 2`; `2 + (-1)*2 = 0` |

Kết quả cuối: `0`.

**Bài 2 (b):** RPN là `3 4 5 - -`. Tính: push 3, 4, 5; `'-'` → `4 - 5 = -1`; `'-'` → `3 - (-1) = 4`. Với `eval_rpn` sai thứ tự: `'-'` đầu cho `5 - 4 = 1`, `'-'` sau cho `1 - 3 = -2`.

Các cell dưới kiểm chứng Bài 1, 2 và chạy đáp án Bài 3–6 với assert.

In [37]:
# Bài 1 — kiểm chứng
print(tokenize("  7-(10 )"))
print(eval_no_paren("8-3-2"))
print(calculate_stack("-(-(4))"))
print(to_rpn(tokenize("5-(1+2)")))
print(eval_no_paren("1-(2+3)"))
assert tokenize("  7-(10 )") == [7, "-", "(", 10, ")"]
assert eval_no_paren("8-3-2") == 3
assert calculate_stack("-(-(4))") == 4
assert to_rpn(tokenize("5-(1+2)")) == [5, 1, 2, "+", "-"]
assert eval_no_paren("1-(2+3)") == 2


# Bài 2 (a) — chụp trạng thái ngay sau mỗi '(' và ')'
def snapshots_at_parens(s: str) -> list:
    snaps = []
    stack = []
    result, sign, num = 0, 1, 0
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append((result, sign))
            result, sign = 0, 1
            snaps.append((i, ch, list(stack), result))
        elif ch == ")":
            result += sign * num
            num = 0
            r, sg = stack.pop()
            result = r + sg * result
            snaps.append((i, ch, list(stack), result))
    return snaps

for i, ch, st, res in snapshots_at_parens("2-(5-(1+2))"):
    print(f"vị trí {i:2} {ch!r}  stack = {str(st):20} result = {res}")
assert snapshots_at_parens("2-(5-(1+2))") == [
    (2, "(", [(2, -1)], 0),
    (5, "(", [(2, -1), (5, -1)], 0),
    (9, ")", [(2, -1)], 2),
    (10, ")", [], 0),
]
assert calculate_stack("2-(5-(1+2))") == 0

# Bài 2 (b)
assert to_rpn(tokenize("3 - (4 - 5)")) == [3, 4, 5, "-", "-"]
assert eval_rpn([3, 4, 5, "-", "-"]) == 4
assert wrong_eval_rpn([3, 4, 5, "-", "-"]) == -2
print("Bài 1 và Bài 2: PASS")

[7, '-', '(', 10, ')']
3
4
[5, 1, 2, '+', '-']
2
vị trí  2 '('  stack = [(2, -1)]            result = 0
vị trí  5 '('  stack = [(2, -1), (5, -1)]   result = 0
vị trí  9 ')'  stack = [(2, -1)]            result = 2
vị trí 10 ')'  stack = []                   result = 0
Bài 1 và Bài 2: PASS


In [38]:
# Bài 3 — điền TODO
def answer_exercise_calculate(s: str) -> int:
    stack = []
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)                      # TODO 1
        elif ch in "+-":
            result += sign * num                          # TODO 2
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append((result, sign))                  # TODO 3
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num                          # TODO 4
            num = 0
            prev_result, prev_sign = stack.pop()
            result = prev_result + prev_sign * result
    return result + sign * num                            # TODO 5


# Bài 4 — đệ quy viết lại
def answer_exercise_recursive(s: str) -> int:
    n = len(s)

    def parse(i: int):
        result, sign, num = 0, 1, 0
        while i < n:
            ch = s[i]
            if ch.isdigit():
                num = num * 10 + int(ch)
            elif ch in "+-":
                result += sign * num
                num = 0
                sign = 1 if ch == "+" else -1
            elif ch == "(":
                num, i = parse(i + 1)
            elif ch == ")":
                return result + sign * num, i
            i += 1
        return result + sign * num, i

    return parse(0)[0]


for f in (answer_exercise_calculate, answer_exercise_recursive):
    assert f("1 + 1") == 2
    assert f(" 2-1 + 2 ") == 3
    assert f("(1+(4+5+2)-3)+(6+8)") == 23
    assert f("1-(     -2)") == 3
    assert f("12+3") == 15
    assert f("-(2 + 3)") == -5
    assert f("1 - (2 - (3 - (4 - 5)))") == 3
    assert f("42") == 42

random.seed(2024)
for _ in range(1000):
    s = rand_expr()
    assert answer_exercise_calculate(s) == answer_exercise_recursive(s) == oracle(s), s
print("Bài 3 và Bài 4: PASS (kể cả 1000 biểu thức ngẫu nhiên so với eval)")

Bài 3 và Bài 4: PASS (kể cả 1000 biểu thức ngẫu nhiên so với eval)


In [39]:
def div_trunc(a: int, b: int) -> int:
    """Chia nguyên làm tròn về 0 (khác // của Python khi trái dấu)."""
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b > 0) else -q

print("-3 // 2 =", -3 // 2, "   nhưng div_trunc(-3, 2) =", div_trunc(-3, 2))


def apply_op(stack: list, op: str, num: int) -> None:
    if op == "+":
        stack.append(num)
    elif op == "-":
        stack.append(-num)
    elif op == "*":
        stack.append(stack.pop() * num)
    else:
        stack.append(div_trunc(stack.pop(), num))


# Bài 5 — LeetCode #227
def answer_calculate_ii(s: str) -> int:
    stack, num, op = [], 0, "+"          # op = toán tử đứng TRƯỚC số đang đọc
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-*/":
            apply_op(stack, op, num)
            num, op = 0, ch
    apply_op(stack, op, num)              # chốt số cuối
    return sum(stack)


# Bài 6 — + - * / và ngoặc (recursive descent)
def answer_calculate_iii(s: str) -> int:
    n = len(s)

    def parse(i: int):
        stack, num, op = [], 0, "+"
        while i < n:
            ch = s[i]
            if ch.isdigit():
                num = num * 10 + int(ch)
            elif ch == "(":
                num, i = parse(i + 1)     # cả nhóm = một con số
            elif ch in "+-*/":
                apply_op(stack, op, num)
                num, op = 0, ch
            elif ch == ")":
                apply_op(stack, op, num)
                return sum(stack), i
            i += 1
        apply_op(stack, op, num)
        return sum(stack), i

    return parse(0)[0]


assert answer_calculate_ii("3+2*2") == 7
assert answer_calculate_ii(" 3/2 ") == 1
assert answer_calculate_ii(" 3+5 / 2 ") == 5
assert answer_calculate_ii("14-3/2") == 13
assert answer_calculate_ii("2*3*4-10/3") == 21

assert answer_calculate_iii("2*(5+5*2)/3+(6/2+8)") == 21
assert answer_calculate_iii("(2+6*3+5-(3*14/7+2)*5)+3") == -12
assert answer_calculate_iii("-(3*2)+10") == 4
assert answer_calculate_iii("2*(-3)") == -6
assert answer_calculate_iii("(1-6)/4") == -1


# Đối chiếu ngẫu nhiên: không có '/' thì eval là trọng tài đúng (+ - * và ngoặc)
def rand_expr_ops(depth: int, ops: str, allow_paren: bool) -> str:
    out = ""
    for j in range(random.randint(1, 4)):
        if j > 0:
            out += " " + random.choice(ops) + " "
        if allow_paren and depth < 2 and random.random() < 0.3:
            term = "(" + rand_expr_ops(depth + 1, ops, allow_paren) + ")"
        else:
            term = str(random.randint(0, 12))
        if allow_paren and j == 0 and random.random() < 0.25:
            term = "-" + term
        out += term
    return out

random.seed(772)
for _ in range(1000):
    s1 = rand_expr_ops(0, "+-*", allow_paren=True)
    assert answer_calculate_iii(s1) == eval(s1), s1
    s2 = rand_expr_ops(0, "+-*", allow_paren=False)
    assert answer_calculate_ii(s2) == answer_calculate_iii(s2) == eval(s2), s2
    s3 = rand_expr_ops(0, "+-*/", allow_paren=False).replace("/ 0", "/ 7")
    assert answer_calculate_ii(s3) == answer_calculate_iii(s3), s3
print("Bài 5 và Bài 6: PASS (kể cả 3000 biểu thức ngẫu nhiên)")

-3 // 2 = -2    nhưng div_trunc(-3, 2) = -1
Bài 5 và Bài 6: PASS (kể cả 3000 biểu thức ngẫu nhiên)


## 25. Template ngắn nên nhớ

```python
class Solution:
    def calculate(self, s: str) -> int:
        stack, result, sign, num = [], 0, 1, 0
        for ch in s:
            if ch.isdigit():
                num = num * 10 + int(ch)          # 1) số thì GHÉP
            elif ch in "+-":
                result += sign * num              # 2) dấu thì CHỐT số, đổi sign
                num, sign = 0, (1 if ch == "+" else -1)
            elif ch == "(":
                stack.append((result, sign))      # 3) mở thì CẤT mức ngoài
                result, sign = 0, 1
            elif ch == ")":
                result += sign * num              # 4) đóng thì chốt rồi GHÉP với mức ngoài
                num = 0
                r, sg = stack.pop()
                result = r + sg * result
        return result + sign * num                # 5) hết thì CHỐT số cuối
```

Câu thần chú: **"Số thì ghép, dấu thì chốt, mở thì cất, đóng thì ghép, hết thì chốt."**

Và ý tưởng gốc cần nhớ: **trừ là cộng số đối** → biểu thức chỉ có `+`, `-` là tổng các hạng tử có dấu; dấu trước ngoặc áp lên **cả nhóm**, nên khi gặp `'('` phải **đóng băng** `(result, sign)` của mức ngoài vào stack.

## 26. Checklist trước khi submit

- Mình có **chốt số cuối** (`return result + sign * num`) chưa? Đã thử một biểu thức **kết thúc bằng số** như `"1 + 1"` chưa?
- Số nhiều chữ số có được **ghép** (`num * 10 + d`) chứ không bị ghi đè? Đã thử `"12+3"` chưa?
- Ở `'('`: cất **cả hai** `result` và `sign`, rồi reset `result = 0`, `sign = +1`?
- Ở `')'`: chốt `num`, **đặt lại `num = 0`**, rồi `result = prev_result + prev_sign * result`?
- Nếu cất hai phần tử rời thay vì tuple: lấy ra có đúng thứ tự **ngược** chưa?
- Dấu trừ một ngôi: đã thử `"-1"`, `"-(2 + 3)"`, `"1-(     -2)"` chưa? (Cách 2 không cần `if` riêng.)
- Khoảng trắng ở đầu, cuối, giữa — đã thử `" 2-1 + 2 "` chưa?
- **Không** dùng `eval` trong code nộp; không dùng đệ quy nếu không chắc về độ sâu lồng ngoặc.